<a href="https://colab.research.google.com/github/financieras/articulos/blob/main/Deep_Learning_Fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Deep Learning Fundamentals**
## **Forward and Backward Propagation explained**

Las redes neuronales han revolucionado el campo de la inteligencia artificial, permitiéndonos resolver problemas que antes parecían imposibles. Pero, ¿cómo aprenden realmente estas redes? En este notebook descubrirás los fundamentos del Deep Learning implementando un perceptrón multicapa desde cero, usando únicamente NumPy. Entenderás cómo fluyen los datos hacia adelante (forward propagation) y cómo el error se propaga hacia atrás (backpropagation) para ajustar los parámetros de la red. Al final, compararemos nuestra implementación con Scikit-learn y aprenderás una valiosa lección: en Machine Learning, más complejo no siempre significa mejor.

---

## **Tabla de Contenido**

1. Introducción
2. Fundamentos
3. Funciones de Activación: La Clave de la No-Linealidad
4. Arquitectura y Hiperparámetros
5. Preparando los Datos: El Dataset Iris
6. Forward Propagation: El Viaje de los Datos
7. Backward Propagation: Aprendiendo del Error
8. Implementación Completa desde Cero
9. Experimentos: Aprendiendo Haciendo
10. Comparación con Scikit-learn
11. Conclusiones

# **1. Introducción**

## **1.1 ¿Qué aprenderás en este notebook?**

Bienvenido a tu primer proyecto de Deep Learning. Si ya has trabajado con regresión lineal o logística usando descenso del gradiente, estás perfectamente preparado para dar este siguiente paso. En este notebook aprenderás:

- **Los bloques fundamentales** de las redes neuronales: neuronas artificiales, pesos y bias
- **Cómo funciona el aprendizaje** a través de forward y backward propagation
- **A implementar una red neuronal completa** usando solo NumPy (¡sin magia, solo matemáticas!)
- **Las funciones de activación** y por qué son esenciales para resolver problemas complejos
- **El equilibrio entre simplicidad y complejidad**: cómo evitar overfitting y underfitting
- **Comparar tu implementación** con librerías profesionales como Scikit-learn

Al finalizar, no solo habrás construido tu propia red neuronal, sino que entenderás profundamente cada línea de código y cada operación matemática detrás del aprendizaje profundo.

---

## **1.2 El mundo del Machine Learning y el Deep Learning**

Antes de sumergirnos en el código, ubiquémonos en el panorama general del Machine Learning.

**Machine Learning** es el campo de la inteligencia artificial que permite a las computadoras aprender patrones a partir de datos, sin ser programadas explícitamente para cada tarea. Dentro de este campo existen diferentes enfoques:

- **Machine Learning Clásico**: Algoritmos como regresión lineal, regresión logística, árboles de decisión, SVM, etc. Son poderosos pero tienen limitaciones con datos muy complejos o de alta dimensionalidad.

- **Deep Learning**: Un subcampo del Machine Learning basado en redes neuronales artificiales con múltiples capas (de ahí el término "profundo"). Estas arquitecturas pueden aprender representaciones jerárquicas de los datos, capturando patrones cada vez más abstractos en cada capa.

**¿Por qué Deep Learning es especial?**

Imagina que quieres enseñar a una computadora a reconocer gatos en imágenes. Con Machine Learning clásico, tendrías que diseñar manualmente las características: "busca orejas puntiagudas", "detecta bigotes", etc. Con Deep Learning, la red aprende automáticamente qué características son importantes, desde bordes básicos en las primeras capas hasta conceptos complejos como "textura de pelaje" o "forma de gato" en las capas más profundas.

**Conexión con lo que ya sabes:**

Si has implementado regresión logística, ya conoces:
- La función sigmoide (una función de activación)
- El cálculo de gradientes
- El descenso del gradiente para actualizar parámetros
- Funciones de pérdida (como log loss)

Una red neuronal es esencialmente **múltiples regresiones logísticas apiladas**, pero con funciones de activación más sofisticadas y una forma sistemática de entrenar todas las capas simultáneamente usando backpropagation. ¡Estás más cerca de lo que crees!

---

## **1.3 Configuración del entorno**

Comencemos importando las librerías que necesitaremos. Utilizaremos principalmente NumPy para los cálculos numéricos, Matplotlib para visualizaciones, y Scikit-learn solo para cargar el dataset Iris y, al final, para comparar resultados.

In [1]:
# Core libraries for numerical computing and data manipulation
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn for dataset and comparison
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Environment configured successfully")
print(f"NumPy version: {np.__version__}")

Environment configured successfully
NumPy version: 2.0.2


**¿Por qué estas librerías?**

- **NumPy**: El núcleo de toda computación científica en Python. Nos permite trabajar eficientemente con matrices y vectores.
- **Matplotlib/Seaborn**: Para crear visualizaciones que nos ayuden a entender qué está aprendiendo nuestra red.
- **Scikit-learn**: Solo para cargar datos y comparar nuestros resultados con una implementación profesional.

Fijamos la semilla aleatoria (`np.random.seed(42)`) para garantizar que nuestros resultados sean reproducibles. Cada vez que ejecutes este notebook, obtendrás los mismos resultados.

---

**¡Perfecto!** Has completado la introducción. Ahora estás listo para sumergirte en los fundamentos de las redes neuronales. En la siguiente sección descubrirás qué es exactamente una neurona artificial y cómo estas pequeñas unidades, trabajando juntas, pueden resolver problemas sorprendentemente complejos.

# **2. Fundamentos**

## **2.1 ¿Qué es una neurona artificial?**

Una **neurona artificial** es una unidad matemática inspirada en las neuronas biológicas del cerebro. Aunque mucho más simple que su contraparte biológica, captura la idea esencial: recibir múltiples señales de entrada, procesarlas, y producir una salida.

**Anatomía de una neurona artificial:**

```
Entradas (x₁, x₂, ..., xₙ)
      ↓
   [Pesos w₁, w₂, ..., wₙ]
      ↓
  Suma ponderada + bias
      ↓
  Función de activación
      ↓
    Salida (a)
```

Matemáticamente, una neurona realiza dos operaciones:

1. **Combinación lineal**: Calcula z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b
2. **Activación no-lineal**: Aplica una función f(z) para obtener la salida a = f(z)

**Ejemplo concreto:**

Imagina que quieres predecir si un estudiante aprobará un examen basándote en:
- x₁ = horas de estudio
- x₂ = asistencia a clases (%)
- x₃ = calificación en examen previo

La neurona podría aprender pesos como:
- w₁ = 0.4 (las horas de estudio son muy importantes)
- w₂ = 0.3 (la asistencia también importa)
- w₃ = 0.5 (el desempeño previo es un buen predictor)
- b = -2.0 (el bias ajusta el umbral de decisión)

Si un estudiante tiene [5 horas, 80% asistencia, 7.5 de nota previa]:
```
z = 0.4(5) + 0.3(80) + 0.5(7.5) - 2.0
z = 2.0 + 24.0 + 3.75 - 2.0 = 27.75
```

Luego, una función de activación convertiría este número en una probabilidad de aprobar.

Veamos esto en código:

In [ ]:
# Simple artificial neuron example
def simple_neuron(inputs, weights, bias, activation='sigmoid'):
    """
    Simulates a single artificial neuron.

    Parameters:
    - inputs: array of input values
    - weights: array of weight values (same length as inputs)
    - bias: bias term
    - activation: activation function ('sigmoid' or 'relu')

    Returns:
    - z: weighted sum
    - output: activated output of the neuron
    """
    # Step 1: Weighted sum
    z = np.dot(weights, inputs) + bias

    # Step 2: Apply activation function
    if activation == 'sigmoid':
        output = 1 / (1 + np.exp(-z))
    elif activation == 'relu':
        output = max(0, z)
    else:
        output = z  # linear activation

    return z, output

# Example: Student passing prediction
inputs = np.array([5, 80, 7.5])  # hours, attendance %, previous grade
weights = np.array([0.4, 0.3, 0.5])
bias = -2.0

z, output = simple_neuron(inputs, weights, bias, activation='sigmoid')

print(f"Inputs: {inputs}")
print(f"Weights: {weights}")
print(f"Bias: {bias}")
print(f"Weighted sum (z): {z:.2f}")
print(f"Activated output: {output:.4f} -> Probability of passing: {output*100:.2f}%")

---

## **2.2 Pesos y bias: Los parámetros aprendibles**

Los **pesos (weights)** y el **bias** son los parámetros que la red neuronal ajusta durante el entrenamiento. Entender su rol es fundamental.

**Los Pesos (w):**

Los pesos determinan la **importancia relativa** de cada entrada.

- Un peso grande (positivo): "Esta característica es muy importante y aumenta la salida"
- Un peso pequeño o cercano a cero: "Esta característica no es relevante"
- Un peso negativo: "Esta característica disminuye la salida"

**Analogía**: Piensa en los pesos como los "controles de volumen" en un mezclador de audio. Cada entrada es un instrumento, y los pesos deciden qué tan fuerte debe sonar cada uno en la mezcla final.

**El Bias (b):**

El bias es un término independiente que **desplaza** la función de activación. Permite que la neurona se active incluso cuando todas las entradas son cero.

- Sin bias: La función pasa por el origen (0,0)
- Con bias positivo: La función se desplaza hacia arriba
- Con bias negativo: La función se desplaza hacia abajo

**Analogía**: Si los pesos son el volumen de cada instrumento, el bias es el volumen maestro de fondo que siempre está presente.

**¿Por qué necesitamos ambos?**

Veamos un ejemplo visual:

In [ ]:
# Visualizing the effect of weights and bias
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x = np.linspace(-5, 5, 100)

# Plot 1: Effect of weights (no bias)
axes[0].plot(x, 0.5 * x, label='w=0.5, b=0', linewidth=2)
axes[0].plot(x, 1.0 * x, label='w=1.0, b=0', linewidth=2)
axes[0].plot(x, 2.0 * x, label='w=2.0, b=0', linewidth=2)
axes[0].set_title('Effect of Weights (no bias)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Input (x)')
axes[0].set_ylabel('Output (wx)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].axvline(0, color='black', linewidth=0.5)

# Plot 2: Effect of bias (fixed weight)
axes[1].plot(x, x - 2, label='w=1, b=-2', linewidth=2)
axes[1].plot(x, x + 0, label='w=1, b=0', linewidth=2)
axes[1].plot(x, x + 2, label='w=1, b=+2', linewidth=2)
axes[1].set_title('Effect of Bias (fixed weight)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Input (x)')
axes[1].set_ylabel('Output (wx + b)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axvline(0, color='black', linewidth=0.5)

# Plot 3: Combined effect with activation
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

axes[2].plot(x, sigmoid(0.5*x + 0), label='w=0.5, b=0', linewidth=2)
axes[2].plot(x, sigmoid(2.0*x + 0), label='w=2.0, b=0', linewidth=2)
axes[2].plot(x, sigmoid(1.0*x - 2), label='w=1.0, b=-2', linewidth=2)
axes[2].plot(x, sigmoid(1.0*x + 2), label='w=1.0, b=+2', linewidth=2)
axes[2].set_title('Combined with Sigmoid', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Input (x)')
axes[2].set_ylabel('Output sigmoid(wx + b)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Weight controls the slope (sensitivity)")
print("Bias controls the shift (threshold)")

**Inicialización de parámetros:**

Un detalle crucial: ¿con qué valores comenzamos los pesos y bias antes del entrenamiento?

- **Inicialización en ceros**: Mala idea. Todas las neuronas aprenderían lo mismo (simetría).
- **Inicialización aleatoria pequeña**: Lo más común. Valores pequeños cerca de cero, pero diferentes entre sí.
- **Bias en cero**: Generalmente se inicia en cero sin problemas.

In [ ]:
# Proper weight initialization
def initialize_weights(n_inputs, n_neurons):
    """
    Initialize weights with small random values and bias as zeros.
    Uses He initialization for better performance with ReLU.

    Parameters:
    - n_inputs: number of input features
    - n_neurons: number of neurons in the layer

    Returns:
    - weights: initialized weight matrix (n_neurons, n_inputs)
    - bias: initialized bias vector (n_neurons, 1)
    """
    # He initialization: multiply by sqrt(2/n_inputs)
    weights = np.random.randn(n_neurons, n_inputs) * np.sqrt(2.0 / n_inputs)
    bias = np.zeros((n_neurons, 1))

    return weights, bias

# Example
W, b = initialize_weights(n_inputs=3, n_neurons=5)

print(f"Weight matrix shape: {W.shape}")  # (5, 3) - 5 neurons, 3 inputs each
print(f"Bias vector shape: {b.shape}")     # (5, 1) - 1 bias per neuron
print(f"\nExample of initialized weights:\n{W}")
print(f"\nInitialized biases:\n{b.T}")

---

## **2.3 De una neurona a una red: El perceptrón multicapa**

Una sola neurona es limitada. Puede aprender relaciones lineales (o casi lineales con activación), pero no puede capturar patrones complejos. **La magia ocurre cuando conectamos múltiples neuronas en capas.**

**Arquitectura de un Perceptrón Multicapa (MLP):**

```
CAPA DE ENTRADA → CAPAS OCULTAS → CAPA DE SALIDA
    (features)      (hidden)         (output)
    
    x₁ ──┐
    x₂ ──┼──→ [Neurona 1] ──┐
    x₃ ──┼──→ [Neurona 2] ──┼──→ [Neurona A] ──→ y₁
    x₄ ──┘      [Neurona 3] ──┘    [Neurona B] ──→ y₂
                                    [Neurona C] ──→ y₃
    
    Input       Hidden Layer       Output Layer
    Layer       (con activación)   (con activación)
```

**Componentes de una red:**

1. **Capa de entrada**: No es realmente una "capa" de neuronas, simplemente los datos crudos (features).

2. **Capas ocultas**: Una o más capas donde ocurre el "pensamiento". Cada neurona en una capa está conectada a todas las neuronas de la capa anterior (fully connected o densa).

3. **Capa de salida**: Produce las predicciones finales. El número de neuronas depende del problema:
   - Clasificación binaria: 1 neurona
   - Clasificación multiclase: n neuronas (una por clase)
   - Regresión: 1 neurona (o más si predices múltiples valores)

**¿Qué hace "profunda" a una red?**

El término "Deep Learning" se refiere a redes con **múltiples capas ocultas** (2 o más). Cada capa aprende representaciones cada vez más abstractas:

- **Capa 1**: Detecta características básicas (bordes, colores en imágenes)
- **Capa 2**: Combina características básicas en patrones (texturas, formas simples)
- **Capa 3**: Reconoce conceptos complejos (ojos, ruedas, etc.)
- **Capa final**: Toma decisiones (¿es un gato? ¿es un coche?)

In [ ]:
# Visualizing a simple MLP architecture
def visualize_mlp_architecture(layer_sizes):
    """
    Visualize the architecture of a Multi-Layer Perceptron.

    Parameters:
    - layer_sizes: list of integers representing neurons in each layer
                   e.g., [4, 5, 3, 3] means 4 inputs, 5 neurons in hidden1,
                   3 in hidden2, and 3 outputs
    """
    fig, ax = plt.subplots(figsize=(12, 6))

    # Calculate positions
    n_layers = len(layer_sizes)
    max_neurons = max(layer_sizes)

    layer_names = ['Input'] + [f'Hidden {i}' for i in range(1, n_layers-1)] + ['Output']

    # Draw neurons
    for layer_idx, (size, name) in enumerate(zip(layer_sizes, layer_names)):
        x = layer_idx * 3
        y_offset = (max_neurons - size) / 2

        for neuron_idx in range(size):
            y = neuron_idx + y_offset
            circle = plt.Circle((x, y), 0.3, color='skyblue', ec='navy', linewidth=2, zorder=3)
            ax.add_patch(circle)

            # Draw connections to next layer
            if layer_idx < n_layers - 1:
                next_size = layer_sizes[layer_idx + 1]
                next_y_offset = (max_neurons - next_size) / 2

                for next_neuron_idx in range(next_size):
                    next_y = next_neuron_idx + next_y_offset
                    ax.plot([x + 0.3, x + 3 - 0.3], [y, next_y],
                           'gray', alpha=0.3, linewidth=0.5, zorder=1)

        # Layer label
        ax.text(x, -1, name, ha='center', fontsize=12, fontweight='bold')

    ax.set_xlim(-1, (n_layers - 1) * 3 + 1)
    ax.set_ylim(-2, max_neurons + 1)
    ax.axis('off')
    ax.set_title('Multi-Layer Perceptron Architecture', fontsize=14, fontweight='bold', pad=20)

    plt.tight_layout()
    plt.show()

# Example: Network for Iris dataset
# 4 features -> 5 hidden neurons -> 3 hidden neurons -> 3 classes
visualize_mlp_architecture([4, 5, 3, 3])

print("Architecture: 4 inputs -> 5 neurons (hidden layer 1) -> 3 neurons (hidden layer 2) -> 3 outputs")
print("Total trainable parameters: (4x5 + 5) + (5x3 + 3) + (3x3 + 3) = 25 + 18 + 12 = 55 parameters")

---

## **2.4 ¿Por qué funcionan las redes neuronales?**

Esta es una pregunta profunda que los investigadores aún estudian, pero podemos entender las ideas clave.

**1. Teorema de Aproximación Universal:**

Matemáticamente, se ha demostrado que una red neuronal con una sola capa oculta (con suficientes neuronas y la activación adecuada) puede aproximar **cualquier función continua** con la precisión deseada. Es como decir que las redes neuronales son "aproximadores universales".

**2. Aprendizaje de representaciones jerárquicas:**

Las capas profundas aprenden automáticamente características útiles de los datos. No necesitas diseñar manualmente qué buscar; la red lo descubre por sí misma durante el entrenamiento.

**Ejemplo conceptual - Reconocimiento de dígitos escritos a mano:**

```
Píxeles crudos → Bordes → Curvas → Partes de números → Números completos
   [Input]      [Capa 1]  [Capa 2]    [Capa 3]         [Output]
```

**3. Optimización a través del gradiente:**

Gracias al algoritmo de backpropagation (que veremos en detalle más adelante), podemos calcular eficientemente cómo cambiar cada peso para reducir el error. Es como descender por una montaña buscando el valle más bajo (mínimo error).

**4. No-linealidad = Poder expresivo:**

Sin funciones de activación no-lineales, una red neuronal profunda sería equivalente a una simple regresión lineal, sin importar cuántas capas tenga. Las activaciones no-lineales (como ReLU) permiten modelar relaciones complejas.

**Demostración simple - El problema XOR:**

El famoso problema XOR no puede ser resuelto por una sola neurona (es no-linealmente separable), pero una red con una capa oculta lo resuelve fácilmente:

In [ ]:
# XOR problem: classic example of why we need hidden layers
def demonstrate_xor_limitation():
    """
    Shows that XOR cannot be solved by a single neuron,
    but can be solved with a hidden layer.
    """
    # XOR truth table
    X_xor = np.array([[0, 0],
                      [0, 1],
                      [1, 0],
                      [1, 1]])
    y_xor = np.array([0, 1, 1, 0])  # XOR output

    # Visualize XOR problem
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Plot 1: XOR is not linearly separable
    colors = ['red' if label == 0 else 'blue' for label in y_xor]
    axes[0].scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=200, edgecolors='black', linewidth=2)
    axes[0].set_xlabel('X1', fontsize=12)
    axes[0].set_ylabel('X2', fontsize=12)
    axes[0].set_title('XOR Problem\n(Not linearly separable)', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-0.5, 1.5)
    axes[0].set_ylim(-0.5, 1.5)

    # Add text annotations
    for i, (x, y, label) in enumerate(zip(X_xor[:, 0], X_xor[:, 1], y_xor)):
        axes[0].text(x, y-0.15, f'XOR={label}', ha='center', fontsize=10, fontweight='bold')

    # Try to draw a line (impossible to separate perfectly)
    x_line = np.linspace(-0.5, 1.5, 100)
    axes[0].plot(x_line, -x_line + 1, 'k--', linewidth=2, alpha=0.5, label='Linear separation attempt')
    axes[0].legend()

    # Plot 2: Solution with hidden layer
    axes[1].text(0.5, 0.5,
                 'Solution: Neural Network with Hidden Layer\n\n'
                 'Input (X1, X2)\n'
                 '    |\n'
                 'Hidden Layer (2 neurons)\n'
                 '    |\n'
                 'Output (XOR)\n\n'
                 'The hidden layer transforms the feature\n'
                 'space, making the problem linearly\n'
                 'separable in that new space.',
                 ha='center', va='center', fontsize=11,
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[1].axis('off')
    axes[1].set_title('Solution with MLP', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print("A single neuron (simple perceptron) CANNOT solve XOR")
    print("A network with a hidden layer CAN solve it")
    print("This illustrates why we need hidden layers for complex problems")

demonstrate_xor_limitation()

**En resumen:**

Las redes neuronales funcionan porque:
1. Pueden aproximar funciones complejas (teorema de aproximación universal)
2. Aprenden automáticamente características útiles (aprendizaje de representaciones)
3. Se optimizan eficientemente (descenso del gradiente + backpropagation)
4. La no-linealidad les da poder expresivo (funciones de activación)

---

**¡Excelente!** Ahora comprendes los bloques fundamentales de las redes neuronales. En la siguiente sección profundizaremos en las **funciones de activación**, el ingrediente secreto que transforma simples sumas ponderadas en sistemas capaces de aprender patrones extraordinariamente complejos.

# **3. Funciones de Activación: La Clave de la No-Linealidad**

## **3.1 El problema de la linealidad**

Imagina que construyes una red neuronal profunda con 5 capas, pero sin funciones de activación. ¿Qué ocurriría?

Matemáticamente, cada capa realiza una transformación lineal:

```
Capa 1: z₁ = W₁x + b₁
Capa 2: z₂ = W₂z₁ + b₂ = W₂(W₁x + b₁) + b₂
Capa 3: z₃ = W₃z₂ + b₃ = W₃(W₂(W₁x + b₁) + b₂) + b₃
...
```

Si expandimos todas estas operaciones, podemos combinar todas las matrices W y todos los bias b en una sola operación lineal:

```
Output = W_total × x + b_total
```

**Resultado**: No importa cuántas capas apiles, una red puramente lineal es equivalente a una regresión lineal simple. Has perdido toda la profundidad y complejidad.

**Demostración numérica:**

In [ ]:
# Demonstrating the linearity problem
def linear_network_collapse():
    """
    Shows that multiple linear layers collapse into a single linear transformation.
    """
    # Create simple input
    x = np.array([[1.0], [2.0], [3.0]])  # 3 features

    # Three linear layers
    W1 = np.random.randn(4, 3)  # 3 -> 4
    b1 = np.random.randn(4, 1)

    W2 = np.random.randn(5, 4)  # 4 -> 5
    b2 = np.random.randn(5, 1)

    W3 = np.random.randn(2, 5)  # 5 -> 2
    b3 = np.random.randn(2, 1)

    # Forward pass through 3 layers (no activation)
    z1 = np.dot(W1, x) + b1
    z2 = np.dot(W2, z1) + b2
    z3 = np.dot(W3, z2) + b3

    print("Output passing through 3 linear layers:")
    print(z3)

    # Equivalent single layer (mathematical collapse)
    W_total = W3 @ W2 @ W1  # Matrix multiplication collapse
    b_total = W3 @ W2 @ b1 + W3 @ b2 + b3

    z_single = np.dot(W_total, x) + b_total

    print("\nOutput with a single equivalent layer:")
    print(z_single)

    print("\nDifference between both (should be ~0):")
    print(np.max(np.abs(z3 - z_single)))

    print("\n[!] Three linear layers = One linear layer")
    print("    Without non-linear activations, depth is useless")

linear_network_collapse()

**La solución: Funciones de Activación No-Lineales**

Introducir una función no-lineal después de cada transformación lineal rompe este colapso:

```
Capa 1: a₁ = f(W₁x + b₁)          [f = función no-lineal]
Capa 2: a₂ = f(W₂a₁ + b₂)
Capa 3: a₃ = f(W₃a₂ + b₃)
```

Ahora estas capas no pueden colapsarse en una sola operación. Cada capa puede aprender transformaciones complejas y la profundidad sí importa.

**Visualización del impacto:**

In [ ]:
# Visualizing linear vs non-linear transformations
def compare_linear_nonlinear():
    """
    Compare how linear and non-linear transformations affect data separation.
    """
    # Generate non-linearly separable data (two concentric circles)
    np.random.seed(42)
    n_samples = 200

    # Inner circle (class 0)
    r1 = np.random.uniform(0, 1, n_samples//2)
    theta1 = np.random.uniform(0, 2*np.pi, n_samples//2)
    x1 = r1 * np.cos(theta1)
    y1 = r1 * np.sin(theta1)

    # Outer circle (class 1)
    r2 = np.random.uniform(1.5, 2, n_samples//2)
    theta2 = np.random.uniform(0, 2*np.pi, n_samples//2)
    x2 = r2 * np.cos(theta2)
    y2 = r2 * np.sin(theta2)

    X = np.vstack([np.column_stack([x1, y1]), np.column_stack([x2, y2])])
    y = np.hstack([np.zeros(n_samples//2), np.ones(n_samples//2)])

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Original data
    axes[0].scatter(X[y==0, 0], X[y==0, 1], c='red', label='Class 0', alpha=0.6, s=30)
    axes[0].scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Class 1', alpha=0.6, s=30)
    axes[0].set_title('Original Data\n(Not linearly separable)', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlabel('X1')
    axes[0].set_ylabel('X2')

    # Linear transformation (just rotation and scaling)
    W_linear = np.array([[1.5, 0.5], [-0.5, 1.5]])
    X_linear = X @ W_linear.T

    axes[1].scatter(X_linear[y==0, 0], X_linear[y==0, 1], c='red', label='Class 0', alpha=0.6, s=30)
    axes[1].scatter(X_linear[y==1, 0], X_linear[y==1, 1], c='blue', label='Class 1', alpha=0.6, s=30)
    axes[1].set_title('Linear Transformation\n(Still not separable)', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlabel('Z1')
    axes[1].set_ylabel('Z2')

    # Non-linear transformation (distance from origin as new feature)
    distance = np.sqrt(X[:, 0]**2 + X[:, 1]**2)
    angle = np.arctan2(X[:, 1], X[:, 0])
    X_nonlinear = np.column_stack([distance, angle])

    axes[2].scatter(X_nonlinear[y==0, 0], X_nonlinear[y==0, 1], c='red', label='Class 0', alpha=0.6, s=30)
    axes[2].scatter(X_nonlinear[y==1, 0], X_nonlinear[y==1, 1], c='blue', label='Class 1', alpha=0.6, s=30)
    axes[2].axvline(x=1.25, color='green', linestyle='--', linewidth=2, label='Decision boundary')
    axes[2].set_title('Non-Linear Transformation\n(Now linearly separable!)', fontsize=12, fontweight='bold')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlabel('Distance from origin')
    axes[2].set_ylabel('Angle')

    plt.tight_layout()
    plt.show()

    print("Observe how non-linear transformation makes separable a problem")
    print("that was impossible for linear transformations")

compare_linear_nonlinear()

---

## **3.2 ReLU: Activación para capas ocultas**

**ReLU** (Rectified Linear Unit) es actualmente la función de activación más popular para capas ocultas en redes profundas. Su definición es sorprendentemente simple:

```
ReLU(z) = max(0, z)
```

Si la entrada es positiva, la deja pasar. Si es negativa, la convierte en cero.

**¿Por qué ReLU es tan efectiva?**

1. **Simplicidad computacional**: Es extremadamente rápida de calcular (solo una comparación).
2. **Evita el problema del gradiente desvanecido**: A diferencia de sigmoide o tanh, no "aplasta" valores grandes.
3. **Sparsity (dispersión)**: Aproximadamente el 50% de las neuronas estarán inactivas (salida = 0), lo que hace la red más eficiente.
4. **Aproximación a funciones no-lineales**: A pesar de su simplicidad, puede aproximar funciones complejas.

**Comparación con otras activaciones:**

In [ ]:
# Visualizing different activation functions
def plot_activation_functions():
    """
    Compare common activation functions: ReLU, Sigmoid, Tanh, Leaky ReLU.
    """
    z = np.linspace(-5, 5, 200)

    # Define activation functions
    relu = np.maximum(0, z)
    sigmoid = 1 / (1 + np.exp(-z))
    tanh = np.tanh(z)
    leaky_relu = np.where(z > 0, z, 0.01 * z)  # Leaky ReLU with alpha=0.01

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # ReLU
    axes[0, 0].plot(z, relu, 'b-', linewidth=2)
    axes[0, 0].axhline(0, color='black', linewidth=0.5)
    axes[0, 0].axvline(0, color='black', linewidth=0.5)
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_title('ReLU: f(z) = max(0, z)', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('z')
    axes[0, 0].set_ylabel('f(z)')
    axes[0, 0].text(2, 1, 'Advantages:\n- Simple and fast\n- No saturation\n- Sparsity',
                    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

    # Sigmoid
    axes[0, 1].plot(z, sigmoid, 'g-', linewidth=2)
    axes[0, 1].axhline(0, color='black', linewidth=0.5)
    axes[0, 1].axvline(0, color='black', linewidth=0.5)
    axes[0, 1].axhline(1, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_title('Sigmoid: f(z) = 1/(1+e^-z)', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('z')
    axes[0, 1].set_ylabel('f(z)')
    axes[0, 1].text(2, 0.3, 'Problems:\n- Vanishing gradients\n- Output not centered at 0',
                    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

    # Tanh
    axes[1, 0].plot(z, tanh, 'r-', linewidth=2)
    axes[1, 0].axhline(0, color='black', linewidth=0.5)
    axes[1, 0].axvline(0, color='black', linewidth=0.5)
    axes[1, 0].axhline(1, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    axes[1, 0].axhline(-1, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_title('Tanh: f(z) = (e^z - e^-z)/(e^z + e^-z)', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('z')
    axes[1, 0].set_ylabel('f(z)')
    axes[1, 0].text(2, -0.5, 'Better than Sigmoid:\n- Centered at 0\nProblem:\n- Vanishing gradients',
                    bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))

    # Leaky ReLU
    axes[1, 1].plot(z, leaky_relu, 'm-', linewidth=2)
    axes[1, 1].axhline(0, color='black', linewidth=0.5)
    axes[1, 1].axvline(0, color='black', linewidth=0.5)
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_title('Leaky ReLU: f(z) = max(0.01z, z)', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('z')
    axes[1, 1].set_ylabel('f(z)')
    axes[1, 1].text(2, 1, 'Improves ReLU:\n- No dead neurons\n- Small gradient\n  for z < 0',
                    bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

    plt.tight_layout()
    plt.show()

    print("ReLU is the default choice for hidden layers in deep networks")
    print("Leaky ReLU is a variant that solves the 'dying neurons' problem")

plot_activation_functions()

**Implementación de ReLU:**

In [ ]:
# ReLU implementation and derivative
def relu(z):
    """
    ReLU activation function.

    Parameters:
    - z: input array

    Returns:
    - activated output (element-wise max(0, z))
    """
    return np.maximum(0, z)

def relu_derivative(z):
    """
    Derivative of ReLU function.

    Returns 1 if z > 0, else 0.
    This is used during backpropagation.

    Parameters:
    - z: input array

    Returns:
    - derivative (1 for positive z, 0 otherwise)
    """
    return (z > 0).astype(float)

# Test ReLU
test_input = np.array([-2.5, -1.0, 0, 0.5, 1.5, 3.0])
print(f"Input:            {test_input}")
print(f"ReLU output:      {relu(test_input)}")
print(f"ReLU derivative:  {relu_derivative(test_input)}")

**El problema de las "neuronas muertas":**

Un efecto secundario de ReLU es que si una neurona siempre recibe valores negativos, su salida será siempre cero y dejará de aprender (gradiente = 0). Esto se llama "neurona muerta". Variantes como Leaky ReLU (que permite un pequeño gradiente negativo) mitigan este problema.

---

## **3.3 Softmax: Probabilidades en la salida**

Para problemas de clasificación multiclase, necesitamos que la capa de salida produzca **probabilidades** para cada clase. Aquí entra **Softmax**.

**Definición matemática:**

Para un vector z = [z₁, z₂, ..., zₖ] (logits de k clases):

```
Softmax(z)ᵢ = e^zᵢ / Σⱼ e^zⱼ
```

**Propiedades clave de Softmax:**

1. Todas las salidas están en el rango [0, 1]
2. La suma de todas las salidas es exactamente 1
3. Valores más grandes en z producen probabilidades más altas
4. Diferencias pequeñas en z se amplifican exponencialmente

**Ejemplo numérico:**

In [ ]:
# Softmax implementation
def softmax(z):
    """
    Softmax activation function.
    Converts logits to probability distribution.

    Parameters:
    - z: input array (logits), shape (n_classes, n_samples)

    Returns:
    - probability distribution, same shape as z
    """
    # Subtract max for numerical stability (prevents overflow)
    exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

# Example: 3 classes, 1 sample
logits = np.array([[2.0],   # Class 0
                   [1.0],   # Class 1
                   [0.1]])  # Class 2

probabilities = softmax(logits)

print("Logits (raw network output):")
print(logits.flatten())
print("\nProbabilities after Softmax:")
print(probabilities.flatten())
print(f"\nSum of probabilities: {np.sum(probabilities):.6f}")
print(f"\nPrediction: Class {np.argmax(probabilities)}")

# Visualize softmax effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Effect of different logit values
z_range = np.linspace(-3, 3, 100)
z_matrix = np.array([z_range, np.zeros_like(z_range), np.zeros_like(z_range)])
probs = softmax(z_matrix)

axes[0].plot(z_range, probs[0], label='Class 0 (variable)', linewidth=2)
axes[0].plot(z_range, probs[1], label='Class 1 (z=0)', linewidth=2)
axes[0].plot(z_range, probs[2], label='Class 2 (z=0)', linewidth=2)
axes[0].set_xlabel('Logit of Class 0', fontsize=11)
axes[0].set_ylabel('Probability', fontsize=11)
axes[0].set_title('How Softmax varies with different logits', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Example with 3 samples
samples_logits = np.array([[2.0, 1.0, 3.0],
                           [1.0, 2.5, 1.5],
                           [0.1, 0.5, 2.0]])
samples_probs = softmax(samples_logits)

x_pos = np.arange(3)
width = 0.25

axes[1].bar(x_pos - width, samples_probs[0], width, label='Class 0', alpha=0.8)
axes[1].bar(x_pos, samples_probs[1], width, label='Class 1', alpha=0.8)
axes[1].bar(x_pos + width, samples_probs[2], width, label='Class 2', alpha=0.8)
axes[1].set_xlabel('Sample', fontsize=11)
axes[1].set_ylabel('Probability', fontsize=11)
axes[1].set_title('Probability distribution (3 samples)', fontsize=12, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(['Sample 1', 'Sample 2', 'Sample 3'])
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n[Softmax normalizes logits to a valid probability distribution]")

---

## **3.4 La función de pérdida: Cross-Entropy**

Una vez que tenemos probabilidades (gracias a Softmax), necesitamos medir qué tan equivocadas son nuestras predicciones. Para clasificación multiclase, usamos **Cross-Entropy Loss** (también llamada Log Loss).

**Definición matemática:**

Para una muestra con etiqueta verdadera y (en formato one-hot) y predicción ŷ (probabilidades de Softmax):

```
Loss = -Σᵢ yᵢ log(ŷᵢ)
```

Como y es one-hot (solo un 1 y el resto 0s), esto se simplifica a:

```
Loss = -log(ŷc)
```

donde c es la clase correcta.

**Intuición:**

- Si predecimos la clase correcta con alta probabilidad (ŷc ≈ 1), la pérdida es baja: -log(1) = 0
- Si predecimos la clase correcta con baja probabilidad (ŷc ≈ 0), la pérdida es alta: -log(0) → ∞

**Implementación:**

In [ ]:
# Cross-Entropy Loss implementation
def cross_entropy_loss(y_true, y_pred):
    """
    Compute cross-entropy loss for multi-class classification.

    Parameters:
    - y_true: true labels in one-hot encoding, shape (n_classes, n_samples)
    - y_pred: predicted probabilities from softmax, same shape

    Returns:
    - average loss across all samples
    """
    # Number of samples
    m = y_true.shape[1]

    # Clip predictions to prevent log(0)
    y_pred_clipped = np.clip(y_pred, 1e-10, 1 - 1e-10)

    # Compute loss
    loss = -np.sum(y_true * np.log(y_pred_clipped)) / m

    return loss

# Example calculation
y_true = np.array([[0], [1], [0]])  # True class is 1 (one-hot encoded)
y_pred_good = np.array([[0.1], [0.8], [0.1]])  # Good prediction
y_pred_bad = np.array([[0.4], [0.3], [0.3]])   # Bad prediction

loss_good = cross_entropy_loss(y_true, y_pred_good)
loss_bad = cross_entropy_loss(y_true, y_pred_bad)

print("True label: Class 1")
print(f"\nGood prediction (p=0.8 for class 1): Loss = {loss_good:.4f}")
print(f"Bad prediction (p=0.3 for class 1):  Loss = {loss_bad:.4f}")

# Visualize how loss changes with prediction confidence
confidence_range = np.linspace(0.01, 0.99, 100)
loss_range = -np.log(confidence_range)

plt.figure(figsize=(10, 6))
plt.plot(confidence_range, loss_range, linewidth=2, color='crimson')
plt.xlabel('Predicted probability for correct class', fontsize=11)
plt.ylabel('Cross-Entropy Loss', fontsize=11)
plt.title('Penalty for incorrect predictions', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='black', linewidth=0.5)
plt.axvline(x=1.0, color='green', linestyle='--', alpha=0.5, label='Perfect prediction')
plt.axvline(x=0.5, color='orange', linestyle='--', alpha=0.5, label='50% confidence')
plt.legend()
plt.tight_layout()
plt.show()

print("\n[Loss grows exponentially when confidence in correct class decreases]")

**¿Por qué Cross-Entropy y no Mean Squared Error?**

Para clasificación, Cross-Entropy es superior a MSE porque:

1. **Penaliza más las predicciones muy incorrectas**: El gradiente es más fuerte cuando la predicción es mala.
2. **Matemáticamente compatible con Softmax**: La derivada combinada de Softmax + Cross-Entropy es extremadamente simple (lo veremos en backpropagation).
3. **Interpretación probabilística**: Mide la divergencia entre dos distribuciones de probabilidad.

---

**Resumen de la sección:**

En esta sección has aprendido:

1. Por qué la no-linealidad es esencial: sin funciones de activación, una red profunda colapsa a una regresión lineal simple.
2. ReLU para capas ocultas: simple, eficiente y efectiva.
3. Softmax para la capa de salida: convierte logits en probabilidades.
4. Cross-Entropy Loss: la función de pérdida estándar para clasificación multiclase.

Con estos componentes, ya tienes todas las piezas necesarias para entender cómo una red neuronal procesa información. En la siguiente sección exploraremos las diferentes arquitecturas de redes y cómo configurar los hiperparámetros para nuestro experimento con el dataset Iris.

# **4. Arquitectura y Hiperparámetros**

## **4.1 Tipos de arquitecturas: Feedforward, CNN, RNN (panorama general)**

Antes de diseñar nuestra red para el dataset Iris, es útil conocer el panorama general de las arquitecturas de redes neuronales. Cada tipo está especializado para diferentes tipos de datos y problemas.

**1. Feedforward Neural Networks (FNN) / Multi-Layer Perceptron (MLP)**

La arquitectura más básica y la que implementaremos en este notebook. Los datos fluyen en una sola dirección: desde la entrada hasta la salida, sin ciclos.

```
Características:
- Flujo unidireccional (input → hidden → output)
- Capas totalmente conectadas (fully connected / dense)
- Ideal para: datos tabulares, clasificación simple, regresión

Estructura:
Input → Dense Layer → Dense Layer → ... → Output
```

**Cuándo usar FNN/MLP:**
- Datos tabulares (como Iris, datos de clientes, métricas financieras)
- Problemas de clasificación o regresión con características numéricas
- Cuando no hay estructura espacial o temporal en los datos

**2. Convolutional Neural Networks (CNN)**

Diseñadas específicamente para datos con estructura espacial, como imágenes. Utilizan operaciones de convolución que detectan patrones locales.

```
Características:
- Capas convolucionales que detectan patrones locales
- Pooling layers que reducen dimensionalidad
- Conservan la estructura espacial
- Ideal para: imágenes, video, señales 2D

Estructura:
Input → Conv → Pool → Conv → Pool → Flatten → Dense → Output
```

**Cuándo usar CNN:**
- Procesamiento de imágenes (clasificación, detección de objetos)
- Reconocimiento de patrones visuales
- Cualquier dato con estructura espacial 2D o 3D

**3. Recurrent Neural Networks (RNN) y sus variantes (LSTM, GRU)**

Diseñadas para datos secuenciales donde el orden importa. Tienen conexiones recurrentes que permiten "memoria" de estados anteriores.

```
Características:
- Conexiones recurrentes (información del pasado afecta al presente)
- Procesan secuencias de longitud variable
- Variantes (LSTM, GRU) solucionan problema de gradientes a largo plazo
- Ideal para: series temporales, texto, audio

Estructura:
Input(t) → RNN Cell → Output(t)
    ↑         ↓
    └─ hidden state ─┘
```

**Cuándo usar RNN/LSTM:**
- Series temporales (predicción de acciones, clima)
- Procesamiento de lenguaje natural (traducción, sentiment analysis)
- Generación de secuencias (música, texto)

**Comparación visual:**

In [ ]:
# Visualizing different neural network architectures
def visualize_architectures():
    """
    Visual comparison of FNN, CNN, and RNN architectures.
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # FNN / MLP
    axes[0].text(0.5, 0.9, 'Feedforward Neural Network (MLP)',
                 ha='center', fontsize=12, fontweight='bold')
    axes[0].text(0.5, 0.7, 'Input Layer\n|\nHidden Layer 1\n|\nHidden Layer 2\n|\nOutput Layer',
                 ha='center', va='center', fontsize=10,
                 bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    axes[0].text(0.5, 0.2, 'Use: Tabular data\nExample: Iris, Housing prices',
                 ha='center', fontsize=9, style='italic')
    axes[0].axis('off')

    # CNN
    axes[1].text(0.5, 0.9, 'Convolutional Neural Network (CNN)',
                 ha='center', fontsize=12, fontweight='bold')
    axes[1].text(0.5, 0.7, 'Input (image)\n|\nConv Layer + ReLU\n|\nPooling Layer\n|\n'
                           'Conv Layer + ReLU\n|\nPooling Layer\n|\nFlatten\n|\nDense Layer\n|\nOutput',
                 ha='center', va='center', fontsize=10,
                 bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
    axes[1].text(0.5, 0.1, 'Use: Spatial data\nExample: Images, Video',
                 ha='center', fontsize=9, style='italic')
    axes[1].axis('off')

    # RNN
    axes[2].text(0.5, 0.9, 'Recurrent Neural Network (RNN)',
                 ha='center', fontsize=12, fontweight='bold')
    axes[2].text(0.5, 0.7, 'Input(t1) -> RNN Cell -> Output(t1)\n       |\n'
                           'Input(t2) -> RNN Cell -> Output(t2)\n       |\n'
                           'Input(t3) -> RNN Cell -> Output(t3)',
                 ha='center', va='center', fontsize=10,
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
    axes[2].text(0.5, 0.2, 'Use: Sequential data\nExample: Text, Time series',
                 ha='center', fontsize=9, style='italic')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    print("For the Iris dataset we will use an MLP (Feedforward)")
    print("because we have tabular data")

visualize_architectures()

**Arquitecturas híbridas y modernas:**

En la práctica, los modelos más avanzados combinan estas arquitecturas:
- **CNN + RNN**: Para video (CNN extrae características espaciales, RNN procesa la secuencia temporal)
- **Transformers**: Reemplazando RNNs en NLP (BERT, GPT)
- **Attention mechanisms**: Permitiendo que el modelo "enfoque" en partes relevantes de los datos

Para nuestro problema de clasificación del dataset Iris, un **MLP simple** es la elección perfecta.

---

## **4.2 Diseñando nuestra red para Iris**

Ahora que entendemos los tipos de arquitecturas, diseñemos específicamente nuestra red para clasificar las flores del dataset Iris.

**Análisis del problema:**

El dataset Iris contiene:
- **150 muestras** de flores
- **4 características numéricas**: longitud del sépalo, ancho del sépalo, longitud del pétalo, ancho del pétalo
- **3 clases**: Iris Setosa, Iris Versicolor, Iris Virginica

**Diseño de la arquitectura:**

```
Input Layer:     4 neuronas (4 características)
                 ↓
Hidden Layer 1:  Variable (configurable)
                 ↓
Hidden Layer 2:  Variable (opcional, configurable)
                 ↓
Output Layer:    3 neuronas + Softmax (3 clases)
```

**Justificación del diseño:**

1. **Capa de entrada (4 neuronas)**: Una por cada característica. No es una capa de cómputo, solo representa los datos.

2. **Capas ocultas (configurables)**:
   - El número de capas y neuronas se define en la configuración
   - Más que las entradas permite capturar combinaciones de características
   - No demasiadas para evitar overfitting en un dataset pequeño
   - La longitud de la lista determina cuántas capas ocultas tendremos

3. **Capa de salida (3 neuronas)**:
   - Una por clase
   - Softmax para obtener probabilidades

**Visualización de nuestra arquitectura:**

In [ ]:
# Detailed architecture visualization for Iris
def visualize_network_architecture(input_size, hidden_neurons, output_size):
    """
    Detailed visualization of the MLP architecture.

    Parameters:
    - input_size: number of input features
    - hidden_neurons: list of neurons per hidden layer (e.g., [5, 3])
    - output_size: number of output classes
    """
    fig, ax = plt.subplots(figsize=(14, 8))

    # Build complete architecture
    all_layers = [input_size] + hidden_neurons + [output_size]

    # Define layer names and labels
    layer_info = []
    layer_info.append({
        'name': 'Input\n(Features)',
        'neurons': input_size,
        'labels': ['Sepal Length', 'Sepal Width', 'Petal Length', 'Petal Width']
    })

    for i, n_neurons in enumerate(hidden_neurons):
        layer_info.append({
            'name': f'Hidden {i+1}\n(ReLU)',
            'neurons': n_neurons,
            'labels': None
        })

    layer_info.append({
        'name': 'Output\n(Softmax)',
        'neurons': output_size,
        'labels': ['Setosa', 'Versicolor', 'Virginica']
    })

    max_neurons = max(all_layers)
    layer_x_positions = [i * 4 for i in range(len(all_layers))]

    # Draw neurons and connections
    for layer_idx, (layer, x_pos) in enumerate(zip(layer_info, layer_x_positions)):
        n_neurons = layer['neurons']
        y_offset = (max_neurons - n_neurons) / 2

        # Draw neurons
        for neuron_idx in range(n_neurons):
            y = neuron_idx + y_offset

            # Color by layer type
            if layer_idx == 0:
                color = 'lightgreen'
            elif layer_idx == len(layer_info) - 1:
                color = 'lightcoral'
            else:
                color = 'lightskyblue'

            circle = plt.Circle((x_pos, y), 0.35, color=color, ec='navy', linewidth=2, zorder=3)
            ax.add_patch(circle)

            # Add labels for input and output
            if layer['labels']:
                label_x = x_pos - 1.8 if layer_idx == 0 else x_pos + 1.8
                ha = 'right' if layer_idx == 0 else 'left'
                ax.text(label_x, y, layer['labels'][neuron_idx],
                       ha=ha, va='center', fontsize=9)

            # Draw connections to next layer
            if layer_idx < len(layer_info) - 1:
                next_n_neurons = layer_info[layer_idx + 1]['neurons']
                next_y_offset = (max_neurons - next_n_neurons) / 2
                next_x = layer_x_positions[layer_idx + 1]

                for next_neuron_idx in range(next_n_neurons):
                    next_y = next_neuron_idx + next_y_offset
                    ax.plot([x_pos + 0.35, next_x - 0.35], [y, next_y],
                           'gray', alpha=0.2, linewidth=0.8, zorder=1)

        # Layer title
        ax.text(x_pos, max_neurons + 0.8, layer['name'],
               ha='center', fontsize=11, fontweight='bold')

        # Add parameter count between layers
        if layer_idx > 0:
            prev_neurons = all_layers[layer_idx - 1]
            curr_neurons = all_layers[layer_idx]
            n_weights = prev_neurons * curr_neurons
            n_biases = curr_neurons
            total_params = n_weights + n_biases

            mid_x = (layer_x_positions[layer_idx - 1] + x_pos) / 2
            ax.text(mid_x, -1.5, f'{total_params} params\n({n_weights}W + {n_biases}b)',
                   ha='center', fontsize=8, style='italic',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax.set_xlim(-3, layer_x_positions[-1] + 3)
    ax.set_ylim(-2.5, max_neurons + 1.5)
    ax.axis('off')
    ax.set_title('MLP Architecture for Iris Classification',
                fontsize=14, fontweight='bold', pad=20)

    # Calculate and display total parameters
    total_params = 0
    for i in range(len(all_layers) - 1):
        total_params += all_layers[i] * all_layers[i+1] + all_layers[i+1]

    ax.text(layer_x_positions[-1] / 2, -2.2,
           f'Total trainable parameters: {total_params}',
           ha='center', fontsize=11, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))

    plt.tight_layout()
    plt.show()

    # Print detailed parameter breakdown
    print(f"Parameters per layer:")
    for i in range(len(all_layers) - 1):
        n_input = all_layers[i]
        n_output = all_layers[i + 1]
        weights = n_input * n_output
        biases = n_output
        total = weights + biases

        layer_name = "Input -> Hidden1" if i == 0 else \
                     f"Hidden{i} -> Hidden{i+1}" if i < len(hidden_neurons) else \
                     f"Hidden{len(hidden_neurons)} -> Output"

        print(f"  {layer_name}: {n_input}x{n_output} + {n_output} = {total} parameters")

    print(f"  TOTAL: {total_params} parameters")

    return total_params

**Conteo de parámetros:**

Entender cuántos parámetros tiene tu red es crucial:

In [ ]:
# Parameter counting function
def count_parameters(architecture):
    """
    Count total trainable parameters in a neural network.

    Parameters:
    - architecture: list of integers, number of neurons in each layer
                   e.g., [4, 5, 3, 3] means 4 inputs, 5 neurons in hidden1,
                   3 in hidden2, and 3 outputs

    Returns:
    - total_params: total number of parameters
    - breakdown: list of dictionaries with details per layer
    """
    total_params = 0
    breakdown = []

    for i in range(len(architecture) - 1):
        n_input = architecture[i]
        n_output = architecture[i + 1]

        weights = n_input * n_output
        biases = n_output
        layer_params = weights + biases

        total_params += layer_params
        breakdown.append({
            'layer': f'Layer {i} -> {i+1}',
            'weights': weights,
            'biases': biases,
            'total': layer_params
        })

    return total_params, breakdown

---

## **4.3 El "arte" de los hiperparámetros: Learning Rate, Epochs, Capas**

Los **hiperparámetros** son configuraciones que defines antes del entrenamiento y que controlan cómo aprende la red. A diferencia de los parámetros (pesos y bias) que se aprenden, los hiperparámetros los eliges tú.

**Principales hiperparámetros:**

**1. Learning Rate (Tasa de Aprendizaje)**

El hiperparámetro más crítico. Controla qué tan grandes son los pasos durante el descenso del gradiente.

```
nuevo_peso = peso_actual - learning_rate × gradiente
```

**Efectos del Learning Rate:**

In [ ]:
# Visualizing learning rate effects
def visualize_learning_rates():
    """
    Show how different learning rates affect convergence.
    """
    # Simple 1D loss function (parabola)
    def loss_function(w):
        return (w - 2) ** 2

    def gradient(w):
        return 2 * (w - 2)

    # Starting point
    w_start = -2
    n_steps = 50

    # Different learning rates
    learning_rates = [0.01, 0.1, 0.5, 1.1]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    w_range = np.linspace(-3, 5, 200)
    loss_range = loss_function(w_range)

    for idx, lr in enumerate(learning_rates):
        ax = axes[idx]

        # Plot loss function
        ax.plot(w_range, loss_range, 'b-', linewidth=2, label='Loss function')

        # Gradient descent trajectory
        w = w_start
        trajectory = [w]

        for step in range(n_steps):
            grad = gradient(w)
            w = w - lr * grad
            trajectory.append(w)

            # Stop if diverging
            if abs(w) > 10:
                break

        # Plot trajectory
        trajectory = np.array(trajectory)
        losses = loss_function(trajectory)
        ax.plot(trajectory, losses, 'ro-', markersize=4, linewidth=1, alpha=0.6, label='Optimization path')
        ax.plot(trajectory[0], losses[0], 'go', markersize=10, label='Start')

        # Optimal point
        ax.axvline(x=2, color='green', linestyle='--', alpha=0.5, label='Optimal')

        ax.set_xlabel('Weight value', fontsize=10)
        ax.set_ylabel('Loss', fontsize=10)
        ax.set_title(f'Learning Rate = {lr}', fontsize=11, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-3, 5)
        ax.set_ylim(-1, 20)

        # Add verdict
        if lr < 0.1:
            verdict = "Very slow - many iterations"
        elif lr <= 0.5:
            verdict = "Good - converges smoothly"
        elif lr < 1.0:
            verdict = "Fast but oscillates"
        else:
            verdict = "Too large - diverges!"

        ax.text(0.5, 0.95, verdict, transform=ax.transAxes,
               ha='center', va='top', fontsize=9,
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

    plt.tight_layout()
    plt.show()

    print("Typical Learning Rates:")
    print("  - Very small (0.001 - 0.01): Slow but stable")
    print("  - Moderate (0.01 - 0.1): Common balance")
    print("  - Large (0.1 - 1.0): Fast but risky")
    print("  - Too large (>1.0): Diverges, doesn't learn")

visualize_learning_rates()

**2. Epochs (Épocas)**

Una época es un pase completo por todo el dataset de entrenamiento. Más épocas = más oportunidades de aprender, pero riesgo de overfitting.

**3. Arquitectura (Número de capas y neuronas)**

- **Más capas**: Pueden aprender patrones más complejos, pero requieren más datos y tiempo
- **Más neuronas**: Mayor capacidad, pero también mayor riesgo de overfitting
- **Regla general**: Empieza simple, aumenta complejidad solo si es necesario

**4. Inicialización de pesos**

- **Random**: Valores aleatorios pequeños
- **Xavier/Glorot**: Buena para sigmoid/tanh
- **He**: Mejor para ReLU (la que usaremos)

**Tabla de recomendaciones:**

In [ ]:
# Hyperparameter recommendations table
import pandas as pd

recommendations = {
    'Hyperparameter': ['Learning Rate', 'Epochs', 'Hidden Layers', 'Neurons per Layer', 'Activation (hidden)', 'Activation (output)', 'Weight Init'],
    'Typical value': ['0.01 - 0.1', '100 - 1000', '1 - 3', '5 - 100', 'ReLU', 'Softmax', 'He/Xavier'],
    'For Iris': ['0.1', '500', '1 - 2', '5 - 10', 'ReLU', 'Softmax', 'He'],
    'Comment': [
        'Adjust based on convergence',
        'Monitor loss curve',
        'Simpler is better for small data',
        'Avoid over-parameterization',
        'Modern standard',
        'For multiclass classification',
        'He for ReLU, Xavier for others'
    ]
}

df_recommendations = pd.DataFrame(recommendations)
print(df_recommendations.to_string(index=False))

---

## **4.4 Configuración inicial de nuestro experimento**

Ahora configuremos los hiperparámetros para nuestros experimentos con Iris. Esta configuración centralizada te permitirá modificar fácilmente los parámetros y re-ejecutar el experimento.

In [ ]:
# ============================================================================
# CENTRALIZED HYPERPARAMETER CONFIGURATION
# ============================================================================
# Edit here to experiment easily. Re-run subsequent cells after changes.

CONFIG = {
    # Model Architecture
    'model': {
        'input_size': 4,                    # Number of features in Iris dataset
        'hidden_neurons': [5, 3],          # List of neurons per hidden layer
                                            # Length determines number of hidden layers
                                            # Example: [5, 3] = 2 hidden layers with 5 and 3 neurons
                                            # Example: [8] = 1 hidden layer with 8 neurons
        'output_size': 3,                   # Number of classes (Setosa, Versicolor, Virginica)
        'activation_hidden': 'relu',        # Activation for hidden layers: 'relu', 'sigmoid', 'tanh'
        'activation_output': 'softmax'      # Activation for output layer (fixed for multiclass)
    },

    # Training Parameters
    'training': {
        'learning_rate': 0.1,               # Step size for gradient descent
        'epochs': 500,                      # Number of complete passes through training data
        'weight_init': 'he'                 # Weight initialization: 'random', 'xavier', 'he'
    },

    # Data Configuration
    'data': {
        'test_size': 0.2,                   # Proportion of data for testing (0.0 - 1.0)
        'random_state': 42                  # Random seed for reproducibility
    }
}

# Quick access variables (extracted from config for convenience)
INPUT_SIZE = CONFIG['model']['input_size']
HIDDEN_NEURONS = CONFIG['model']['hidden_neurons']
OUTPUT_SIZE = CONFIG['model']['output_size']
NUM_HIDDEN_LAYERS = len(HIDDEN_NEURONS)

LEARNING_RATE = CONFIG['training']['learning_rate']
EPOCHS = CONFIG['training']['epochs']
WEIGHT_INIT = CONFIG['training']['weight_init']

TEST_SIZE = CONFIG['data']['test_size']
RANDOM_SEED = CONFIG['data']['random_state']

# Set global random seed
np.random.seed(RANDOM_SEED)

# Display current configuration
print("=" * 70)
print("EXPERIMENT CONFIGURATION")
print("=" * 70)
print(f"\nModel Architecture:")
print(f"  Input Layer:           {INPUT_SIZE} neurons")
for i, n_neurons in enumerate(HIDDEN_NEURONS, 1):
    print(f"  Hidden Layer {i}:         {n_neurons} neurons (ReLU)")
print(f"  Output Layer:          {OUTPUT_SIZE} neurons (Softmax)")
print(f"  Total Hidden Layers:   {NUM_HIDDEN_LAYERS}")

print(f"\nTraining Parameters:")
print(f"  Learning Rate:         {LEARNING_RATE}")
print(f"  Epochs:                {EPOCHS}")
print(f"  Weight Initialization: {WEIGHT_INIT}")

print(f"\nData Configuration:")
print(f"  Test Split:            {TEST_SIZE * 100:.0f}%")
print(f"  Train Split:           {(1-TEST_SIZE) * 100:.0f}%")
print(f"  Random Seed:           {RANDOM_SEED}")

# Calculate total parameters
architecture = [INPUT_SIZE] + HIDDEN_NEURONS + [OUTPUT_SIZE]
total_params, breakdown = count_parameters(architecture)

print(f"\nParameter Count:")
print(f"  Total Parameters:      {total_params}")
print(f"  Architecture:          {' -> '.join(map(str, architecture))}")

print("=" * 70)
print("\nConfiguration loaded successfully!")
print("To experiment with different settings:")
print("  1. Modify the CONFIG dictionary above")
print("  2. Re-run this cell")
print("  3. Re-run subsequent training cells")
print("=" * 70)

**Visualización de la arquitectura configurada:**

In [ ]:
# Visualize the configured architecture
print("\nVisualizing configured network architecture...\n")
total_params = visualize_network_architecture(
    INPUT_SIZE,
    HIDDEN_NEURONS,
    OUTPUT_SIZE
)

print(f"\nWith 150 samples and {total_params} parameters:")
print(f"  Samples per parameter: {150/total_params:.2f}")
print(f"  Rule of thumb: aim for at least 10 samples per parameter")

if 150/total_params < 10:
    print(f"  WARNING: Risk of overfitting! Consider reducing network complexity")
else:
    print(f"  GOOD: Sufficient data for this architecture")

**Ejemplos de configuraciones alternativas:**

In [ ]:
# Examples of alternative configurations
print("\n" + "=" * 70)
print("ALTERNATIVE CONFIGURATION EXAMPLES")
print("=" * 70)

print("\nExample 1: Shallow Network (1 hidden layer)")
print("  'hidden_neurons': [8]")
print("  -> Simpler, less prone to overfitting")
print("  -> Good baseline for small datasets")

print("\nExample 2: Deep Network (3 hidden layers)")
print("  'hidden_neurons': [10, 8, 5]")
print("  -> More capacity to learn complex patterns")
print("  -> Requires more data to avoid overfitting")

print("\nExample 3: Wide Network (1 large hidden layer)")
print("  'hidden_neurons': [20]")
print("  -> Single layer with many neurons")
print("  -> Can approximate complex functions")

print("\nExample 4: Conservative (very simple)")
print("  'hidden_neurons': [4]")
print("  -> Minimal architecture")
print("  -> Best for very small datasets or simple patterns")

print("\nCurrent configuration:")
print(f"  'hidden_neurons': {HIDDEN_NEURONS}")
print(f"  -> {NUM_HIDDEN_LAYERS} hidden layer(s) with architecture: " +
      ' -> '.join(map(str, architecture)))
print("=" * 70)

**Experimentos planificados:**

In [ ]:
# Planning experiments with different configurations
planned_experiments = {
    'Experiment': [
        'A: Current Config',
        'B: Shallow Network',
        'C: Single Wide Layer',
        'D: Scikit-learn Baseline'
    ],
    'Architecture': [
        ' -> '.join(map(str, architecture)),
        '4 -> 5 -> 3',
        '4 -> 10 -> 3',
        'Default MLP'
    ],
    'Hidden Layers': [
        NUM_HIDDEN_LAYERS,
        1,
        1,
        'Auto'
    ],
    'Parameters': [
        total_params,
        (4*5+5) + (5*3+3),
        (4*10+10) + (10*3+3),
        'Unknown'
    ],
    'Purpose': [
        'Current configuration',
        'Test if simpler is better',
        'Test wider vs deeper',
        'Professional benchmark'
    ]
}

df_experiments = pd.DataFrame(planned_experiments)
print("\nPlanned Experiments:")
print(df_experiments.to_string(index=False))

print("\nHypothesis:")
print("  For a simple dataset like Iris, a shallow network may perform")
print("  as well or better than a deep network, with less overfitting risk.")
print("=" * 70)

---

**Resumen de la sección:**

En esta sección has aprendido:

1. **Tipos de arquitecturas**: FNN/MLP para datos tabulares, CNN para imágenes, RNN para secuencias
2. **Diseño flexible para Iris**: Una arquitectura configurable mediante diccionario
3. **Hiperparámetros críticos**: Learning rate, epochs, arquitectura, inicialización
4. **Configuración centralizada**: Sistema fácil de modificar para experimentación

**Ventajas de la configuración centralizada:**

- Todos los hiperparámetros en un solo lugar
- Fácil experimentación cambiando valores
- El número de capas ocultas se determina automáticamente por la longitud de la lista
- Documentación clara de cada parámetro
- Validación y visualización automática de la arquitectura

Ahora que tenemos clara la arquitectura y los hiperparámetros, es momento de preparar los datos. En la siguiente sección cargaremos el dataset Iris, lo exploraremos, y lo prepararemos para el entrenamiento.

# **5. Preparando los Datos: El Dataset Iris**

## **5.1 Carga y exploración**

El dataset Iris es uno de los conjuntos de datos más famosos en Machine Learning. Fue introducido por el estadístico Ronald Fisher en 1936 y contiene mediciones de flores de iris de tres especies diferentes.

**Cargando el dataset:**

In [ ]:
# Load the Iris dataset
iris = load_iris()

# Extract features and labels
X = iris.data          # Features: sepal length, sepal width, petal length, petal width
y = iris.target        # Labels: 0 (Setosa), 1 (Versicolor), 2 (Virginica)

# Get feature names and target names
feature_names = iris.feature_names
target_names = iris.target_names

print("Iris Dataset loaded successfully")
print("=" * 70)
print(f"Number of samples:     {X.shape[0]}")
print(f"Number of features:    {X.shape[1]}")
print(f"Number of classes:     {len(target_names)}")
print(f"\nFeature names:")
for i, name in enumerate(feature_names):
    print(f"  {i}: {name}")
print(f"\nClass names:")
for i, name in enumerate(target_names):
    print(f"  {i}: {name}")
print("=" * 70)

**Explorando las primeras muestras:**

In [ ]:
# Display first samples
print("\nFirst 5 samples from the dataset:")
print("-" * 80)
print(f"{'Sepal L':>10} {'Sepal W':>10} {'Petal L':>10} {'Petal W':>10} {'Class':>15}")
print("-" * 80)

for i in range(5):
    print(f"{X[i, 0]:>10.1f} {X[i, 1]:>10.1f} {X[i, 2]:>10.1f} {X[i, 3]:>10.1f} {target_names[y[i]]:>15}")

print("-" * 80)

**Estadísticas descriptivas:**

In [ ]:
# Statistical summary
print("\nDescriptive statistics of features:")
print("=" * 80)

stats = {
    'Feature': feature_names,
    'Min': [f"{X[:, i].min():.2f}" for i in range(X.shape[1])],
    'Max': [f"{X[:, i].max():.2f}" for i in range(X.shape[1])],
    'Mean': [f"{X[:, i].mean():.2f}" for i in range(X.shape[1])],
    'Std': [f"{X[:, i].std():.2f}" for i in range(X.shape[1])]
}

df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))

# Class distribution
print("\nClass distribution:")
unique, counts = np.unique(y, return_counts=True)
for class_idx, count in zip(unique, counts):
    print(f"  {target_names[class_idx]:15}: {count} samples ({count/len(y)*100:.1f}aksha)")

print("\nDataset is perfectly balanced (50 samples per class)")
print("=" * 80)

**Visualización del dataset:**

In [ ]:
# Visualize the Iris dataset
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Color map for classes
colors = ['red', 'blue', 'green']
class_colors = [colors[label] for label in y]

# Plot 1: Sepal measurements
axes[0, 0].scatter(X[:, 0], X[:, 1], c=class_colors, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[0, 0].set_xlabel(feature_names[0], fontsize=10)
axes[0, 0].set_ylabel(feature_names[1], fontsize=10)
axes[0, 0].set_title('Sepal Length vs Sepal Width', fontsize=11, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Petal measurements
axes[0, 1].scatter(X[:, 2], X[:, 3], c=class_colors, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[0, 1].set_xlabel(feature_names[2], fontsize=10)
axes[0, 1].set_ylabel(feature_names[3], fontsize=10)
axes[0, 1].set_title('Petal Length vs Petal Width', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Sepal Length vs Petal Length
axes[1, 0].scatter(X[:, 0], X[:, 2], c=class_colors, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[1, 0].set_xlabel(feature_names[0], fontsize=10)
axes[1, 0].set_ylabel(feature_names[2], fontsize=10)
axes[1, 0].set_title('Sepal Length vs Petal Length', fontsize=11, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Feature distributions by class
feature_idx = 2  # Petal length (most discriminative)
for class_idx, class_name in enumerate(target_names):
    class_data = X[y == class_idx, feature_idx]
    axes[1, 1].hist(class_data, bins=15, alpha=0.6, label=class_name, color=colors[class_idx])

axes[1, 1].set_xlabel(feature_names[feature_idx], fontsize=10)
axes[1, 1].set_ylabel('Frequency', fontsize=10)
axes[1, 1].set_title('Distribution of Petal Length by Class', fontsize=11, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Add legend to first plot
legend_elements = [plt.Line2D([0], [0], marker='o', color='w',
                             markerfacecolor=colors[i], markersize=8,
                             label=target_names[i]) for i in range(3)]
axes[0, 0].legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("  - Setosa (red) is clearly separable from the other two classes")
print("  - Versicolor (blue) and Virginica (green) have some overlap")
print("  - Petal length and petal width are the most discriminative features")

**Análisis de correlación:**

In [ ]:
# Correlation matrix
correlation_matrix = np.corrcoef(X.T)

plt.figure(figsize=(10, 8))
im = plt.imshow(correlation_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)

# Add colorbar
cbar = plt.colorbar(im)
cbar.set_label('Correlation coefficient', fontsize=10)

# Set ticks and labels
plt.xticks(range(len(feature_names)), feature_names, rotation=45, ha='right')
plt.yticks(range(len(feature_names)), feature_names)

# Add correlation values
for i in range(len(feature_names)):
    for j in range(len(feature_names)):
        text = plt.text(j, i, f'{correlation_matrix[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=10)

plt.title('Correlation Matrix between Features', fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\nCorrelation interpretation:")
print("  - Petal length and petal width are highly correlated (0.96)")
print("  - This suggests both measure similar aspects of the flower")
print("  - Sepal features have lower correlation with petal features")

---

## **5.2 One-Hot Encoding de las etiquetas**

Actualmente nuestras etiquetas están codificadas como números enteros (0, 1, 2). Para usar Softmax en la capa de salida, necesitamos convertirlas a formato **one-hot encoding**.

**¿Qué es One-Hot Encoding?**

Es una representación donde cada clase se codifica como un vector binario:

```
Clase 0 (Setosa):     [1, 0, 0]
Clase 1 (Versicolor): [0, 1, 0]
Clase 2 (Virginica):  [0, 0, 1]
```

**¿Por qué necesitamos One-Hot Encoding?**

1. **Compatible con Softmax**: Softmax produce un vector de probabilidades; necesitamos compararlo con un vector similar
2. **Cross-Entropy Loss**: La función de pérdida espera vectores de probabilidad para ambas predicciones y etiquetas verdaderas
3. **No implica orden**: Los números 0, 1, 2 podrían sugerir que Virginica > Versicolor > Setosa, lo cual no tiene sentido

**Implementación:**

In [ ]:
# One-Hot Encoding function
def one_hot_encode(labels, num_classes):
    """
    Convert integer labels to one-hot encoded vectors.

    Parameters:
    - labels: array of integer labels, shape (n_samples,)
    - num_classes: number of classes

    Returns:
    - one_hot: array of one-hot vectors, shape (num_classes, n_samples)
    """
    n_samples = labels.shape[0]
    one_hot = np.zeros((num_classes, n_samples))

    # Set the appropriate index to 1 for each sample
    for i, label in enumerate(labels):
        one_hot[label, i] = 1

    return one_hot

# Apply one-hot encoding
y_one_hot = one_hot_encode(y, OUTPUT_SIZE)

print("One-Hot Encoding transformation:")
print("=" * 70)
print(f"Original shape of y:       {y.shape}")
print(f"Shape after one-hot:       {y_one_hot.shape}")
print(f"\nTransformation examples:")
print("-" * 70)

for i in range(5):
    original_label = y[i]
    one_hot_vector = y_one_hot[:, i]
    class_name = target_names[original_label]
    print(f"Sample {i}: Class {original_label} ({class_name:15}) -> {one_hot_vector}")

print("-" * 70)

# Verify the encoding
print("\nVerification:")
print(f"  - Each column sums to 1: {np.all(y_one_hot.sum(axis=0) == 1)}")
print(f"  - Only contains 0s and 1s: {np.all((y_one_hot == 0) | (y_one_hot == 1))}")
print("=" * 70)

**Visualización del One-Hot Encoding:**

In [ ]:
# Visualize one-hot encoding
fig, ax = plt.subplots(figsize=(12, 6))

# Show first 30 samples
n_samples_to_show = 30
one_hot_subset = y_one_hot[:, :n_samples_to_show]

# Create heatmap
im = ax.imshow(one_hot_subset, cmap='RdYlGn', aspect='auto', interpolation='nearest')

# Set ticks
ax.set_yticks(range(OUTPUT_SIZE))
ax.set_yticklabels(target_names)
ax.set_xlabel('Sample Index', fontsize=11)
ax.set_ylabel('Class', fontsize=11)
ax.set_title('One-Hot Encoding Visualization (first 30 samples)',
            fontsize=12, fontweight='bold')

# Add grid
ax.set_xticks(np.arange(n_samples_to_show) - 0.5, minor=True)
ax.set_yticks(np.arange(OUTPUT_SIZE) - 0.5, minor=True)
ax.grid(which='minor', color='white', linewidth=1)

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Value', fontsize=10)

plt.tight_layout()
plt.show()

print("Each column represents one sample with exactly one '1' indicating its class")

---

## **5.3 Normalización y división train/test**

**¿Por qué normalizar?**

Las características en Iris tienen diferentes escalas:
- Sepal width: ~2-4.5 cm
- Petal length: ~1-7 cm

Sin normalización, características con valores grandes dominarían el aprendizaje. La normalización pone todas las características en la misma escala.

**Método: Standardization (Z-score normalization)**

```
x_normalized = (x - mean) / std
```

Esto transforma los datos para tener media 0 y desviación estándar 1.

**Implementación:**

In [ ]:
# Split data first (important: normalize AFTER splitting!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

print("Data splitting:")
print("=" * 70)
print(f"Training samples:   {X_train.shape[0]} ({(1-TEST_SIZE)*100:.0f}aksha)")
print(f"Testing samples:    {X_test.shape[0]} ({TEST_SIZE*100:.0f}aksha)")
print(f"\nClass distribution in training set:")

unique_train, counts_train = np.unique(y_train, return_counts=True)
for class_idx, count in zip(unique_train, counts_train):
    print(f"  {target_names[class_idx]:15}: {count} samples")

print(f"\nClass distribution in test set:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for class_idx, count in zip(unique_test, counts_test):
    print(f"  {target_names[class_idx]:15}: {count} samples")

print("\nNote: stratify=y ensures balanced class distribution in both sets")
print("=" * 70)

**Aplicar normalización:**

In [ ]:
# Normalize the data
scaler = StandardScaler()

# Fit on training data only (to avoid data leakage)
X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)

print("\n" + "=" * 70)
print("Normalization applied:")
print("=" * 70)

print("\nTraining set statistics (after normalization):")
print(f"Mean per feature:             {X_train_normalized.mean(axis=0)}")
print(f"Std deviation per feature:    {X_train_normalized.std(axis=0)}")

print("\nComparison before and after (first feature):")
print("-" * 70)
print(f"{'Statistic':<15} {'Before':<15} {'After':<15}")
print("-" * 70)
print(f"{'Mean':<15} {X_train[:, 0].mean():<15.4f} {X_train_normalized[:, 0].mean():<15.4f}")
print(f"{'Std':<15} {X_train[:, 0].std():<15.4f} {X_train_normalized[:, 0].std():<15.4f}")
print(f"{'Min':<15} {X_train[:, 0].min():<15.4f} {X_train_normalized[:, 0].min():<15.4f}")
print(f"{'Max':<15} {X_train[:, 0].max():<15.4f} {X_train_normalized[:, 0].max():<15.4f}")
print("-" * 70)
print("=" * 70)

**Visualización del efecto de la normalización:**

In [ ]:
# Visualize normalization effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before normalization
axes[0].boxplot([X_train[:, i] for i in range(4)], labels=feature_names)
axes[0].set_ylabel('Value', fontsize=11)
axes[0].set_title('Features BEFORE Normalization', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=45)

# After normalization
axes[1].boxplot([X_train_normalized[:, i] for i in range(4)], labels=feature_names)
axes[1].set_ylabel('Normalized Value', fontsize=11)
axes[1].set_title('Features AFTER Normalization', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\nObserve how all features now have:")
print("  - Mean close to 0 (red dashed line)")
print("  - Similar scale (comparable ranges)")
print("  - This facilitates neural network learning")

**Preparar datos en formato correcto para la red:**

In [ ]:
# Transpose data to match our network's expected format: (n_features, n_samples)
X_train_final = X_train_normalized.T  # Shape: (4, 120)
X_test_final = X_test_normalized.T    # Shape: (4, 30)

# One-hot encode labels
y_train_one_hot = one_hot_encode(y_train, OUTPUT_SIZE)  # Shape: (3, 120)
y_test_one_hot = one_hot_encode(y_test, OUTPUT_SIZE)    # Shape: (3, 30)

print("\n" + "=" * 70)
print("Final data ready for neural network:")
print("=" * 70)
print(f"\nTraining set:")
print(f"  X_train_final shape:     {X_train_final.shape} (features x samples)")
print(f"  y_train_one_hot shape:   {y_train_one_hot.shape} (classes x samples)")

print(f"\nTest set:")
print(f"  X_test_final shape:      {X_test_final.shape} (features x samples)")
print(f"  y_test_one_hot shape:    {y_test_one_hot.shape} (classes x samples)")

print("\nData preparation complete! Ready for training.")
print("=" * 70)

**Importante: Data Leakage**

In [ ]:
# Demonstration of why we fit scaler only on training data
print("\n" + "=" * 70)
print("IMPORTANT: Avoiding Data Leakage")
print("=" * 70)
print("\nWRONG approach (Data Leakage):")
print("  1. Normalize entire dataset")
print("  2. Split into train/test")
print("  -> Test set 'contaminates' training statistics")

print("\nCORRECT approach (No Data Leakage):")
print("  1. Split into train/test")
print("  2. Calculate mean/std ONLY from training set")
print("  3. Apply those statistics to both sets")
print("  -> Test set is truly 'unseen' during training")

print("\nIn our code:")
print("  scaler.fit_transform(X_train)  -> Computes and applies normalization")
print("  scaler.transform(X_test)       -> Only applies (uses train stats)")
print("=" * 70)

**Verificación final de los datos:**

In [ ]:
# Final data verification
print("\n" + "=" * 70)
print("FINAL DATA VERIFICATION")
print("=" * 70)

# Check shapes
print("\nShape verification:")
print(f"  X_train_final:    {X_train_final.shape} == (4, 120)")
print(f"  y_train_one_hot:  {y_train_one_hot.shape} == (3, 120)")
print(f"  X_test_final:     {X_test_final.shape} == (4, 30)")
print(f"  y_test_one_hot:   {y_test_one_hot.shape} == (3, 30)")

# Check normalization
print(f"\nNormalization check (train set):")
print(f"  Mean close to 0:  {np.allclose(X_train_final.mean(axis=1), 0, atol=1e-10)}")
print(f"  Std close to 1:   {np.allclose(X_train_final.std(axis=1), 1, atol=0.1)}")

# Check one-hot encoding
print(f"\nOne-hot encoding check:")
print(f"  Each sample sums to 1 (train): {np.all(y_train_one_hot.sum(axis=0) == 1)}")
print(f"  Each sample sums to 1 (test):  {np.all(y_test_one_hot.sum(axis=0) == 1)}")

# Display sample
print(f"\nSample from training set:")
print(f"  Features (normalized): {X_train_final[:, 0]}")
print(f"  Label (one-hot):       {y_train_one_hot[:, 0]} -> Class {np.argmax(y_train_one_hot[:, 0])}")

print("\nAll checks passed! Data is ready for training.")
print("=" * 70)

---

**Resumen de la sección:**

En esta sección has aprendido:

1. **Exploración del dataset Iris**: 150 muestras, 4 características, 3 clases balanceadas
2. **One-Hot Encoding**: Conversión de etiquetas enteras a vectores binarios para compatibilidad con Softmax
3. **Normalización**: Estandarización de características para facilitar el aprendizaje
4. **División train/test**: 80/20 con estratificación para mantener distribución de clases
5. **Data leakage**: Por qué es crítico normalizar después de dividir los datos

Los datos están ahora perfectamente preparados para entrenar nuestra red neuronal. En la siguiente sección, exploraremos en detalle cómo funciona el **Forward Propagation**: el proceso mediante el cual los datos viajan a través de la red para producir predicciones.

# **6. Forward Propagation: El Viaje de los Datos**

## **6.1 Visualización del flujo capa por capa**

El **Forward Propagation** es el proceso mediante el cual los datos de entrada viajan a través de la red neuronal para producir una predicción. Es el "viaje hacia adelante" que da nombre al algoritmo.

**El flujo completo:**

```
Input (X) → Hidden Layer 1 → Hidden Layer 2 → ... → Output Layer → Prediction
   ↓              ↓                ↓                      ↓
   4 features    ReLU            ReLU                 Softmax
                                                    Probabilities
```

**Visualización detallada del proceso:**

In [ ]:
# Visualize the forward propagation flow
def visualize_forward_flow(hidden_neurons):
    """
    Visual representation of data flow through the network.

    Parameters:
    - hidden_neurons: list of neurons per hidden layer
    """
    fig, ax = plt.subplots(figsize=(16, 8))

    # Build stages dynamically based on configuration
    stages = []

    # Input stage
    stages.append({
        'name': 'Input\nX',
        'pos': 0,
        'color': 'lightgreen',
        'equation': f'X\n({INPUT_SIZE}, m)',
        'description': 'Normalized features'
    })

    # Hidden layers
    pos = 2
    for i, n_neurons in enumerate(hidden_neurons, 1):
        # Linear transformation
        stages.append({
            'name': f'Layer {i}\nLinear',
            'pos': pos,
            'color': 'lightskyblue',
            'equation': f'Z{i} = W{i}A{i-1} + b{i}\n({n_neurons}, m)',
            'description': 'Linear combination'
        })
        pos += 1.5

        # Activation
        stages.append({
            'name': 'ReLU',
            'pos': pos,
            'color': 'lightblue',
            'equation': f'A{i} = ReLU(Z{i})\n({n_neurons}, m)',
            'description': 'Non-linear activation'
        })
        pos += 2

    # Output layer
    out_idx = len(hidden_neurons) + 1
    stages.append({
        'name': 'Output\nLinear',
        'pos': pos,
        'color': 'lightskyblue',
        'equation': f'Z{out_idx} = W{out_idx}A{len(hidden_neurons)} + b{out_idx}\n({OUTPUT_SIZE}, m)',
        'description': 'Linear combination'
    })
    pos += 1.5

    stages.append({
        'name': 'Softmax',
        'pos': pos,
        'color': 'lightcoral',
        'equation': f'Y_pred = Softmax(Z{out_idx})\n({OUTPUT_SIZE}, m)',
        'description': 'Probabilities'
    })

    # Draw stages
    for stage in stages:
        # Main box
        rect = plt.Rectangle((stage['pos']-0.4, 2), 0.8, 1.5,
                            facecolor=stage['color'], edgecolor='navy', linewidth=2)
        ax.add_patch(rect)

        # Stage name
        ax.text(stage['pos'], 2.75, stage['name'],
               ha='center', va='center', fontsize=10, fontweight='bold')

        # Equation
        ax.text(stage['pos'], 1.2, stage['equation'],
               ha='center', va='center', fontsize=9,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        # Description
        ax.text(stage['pos'], 0.3, stage['description'],
               ha='center', va='center', fontsize=8, style='italic')

    # Draw arrows between stages
    for i in range(len(stages) - 1):
        x_start = stages[i]['pos'] + 0.4
        x_end = stages[i+1]['pos'] - 0.4
        ax.annotate('', xy=(x_end, 2.75), xytext=(x_start, 2.75),
                   arrowprops=dict(arrowstyle='->', lw=2, color='black'))

    # Add parameter labels for weight matrices
    param_y = 4
    param_positions = []

    # Parameters between input and first hidden
    param_positions.append({
        'pos': 1,
        'text': f'W1, b1\n({hidden_neurons[0]}x{INPUT_SIZE}, {hidden_neurons[0]}x1)'
    })

    # Parameters between hidden layers
    for i in range(len(hidden_neurons) - 1):
        pos_x = 2 + (i+1) * 3.5
        param_positions.append({
            'pos': pos_x,
            'text': f'W{i+2}, b{i+2}\n({hidden_neurons[i+1]}x{hidden_neurons[i]}, {hidden_neurons[i+1]}x1)'
        })

    # Parameters between last hidden and output
    last_pos = 2 + len(hidden_neurons) * 3.5
    param_positions.append({
        'pos': last_pos,
        'text': f'W{len(hidden_neurons)+1}, b{len(hidden_neurons)+1}\n({OUTPUT_SIZE}x{hidden_neurons[-1]}, {OUTPUT_SIZE}x1)'
    })

    for label in param_positions:
        ax.text(label['pos'], param_y, label['text'],
               ha='center', va='center', fontsize=8,
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

    max_pos = stages[-1]['pos'] + 1
    ax.set_xlim(-1, max_pos)
    ax.set_ylim(0, 4.5)
    ax.axis('off')
    ax.set_title('Forward Propagation: Data Flow Through the Network',
                fontsize=13, fontweight='bold', pad=20)

    # Add note
    ax.text(max_pos/2, -0.3, 'Note: m = number of samples (batch size)',
           ha='center', fontsize=9, style='italic',
           bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

    plt.tight_layout()
    plt.show()

# Visualize with current configuration
visualize_forward_flow(HIDDEN_NEURONS)

print("At each step:")
print("  1. Linear transformation: Z = WA_prev + b")
print("  2. Non-linear activation: A = f(Z)")
print("  3. The result A becomes the input for the next layer")

In [ ]:
# Demonstrate dimensionality through the network
def show_dimensions(input_size, hidden_neurons, output_size, batch_size=10):
    """
    Show how matrix dimensions change through forward propagation.

    Parameters:
    - input_size: number of input features
    - hidden_neurons: list of neurons per hidden layer
    - output_size: number of output classes
    - batch_size: number of samples (for demonstration)
    """
    print("=" * 80)
    print(f"DIMENSIONS THROUGH THE NETWORK (example with {batch_size} samples)")
    print("=" * 80)

    dimensions = []

    # Input
    dimensions.append({
        'Stage': 'Input X',
        'Shape': f'({input_size}, {batch_size})',
        'Description': 'Features x Samples'
    })

    dimensions.append({
        'Stage': '─────────',
        'Shape': '─────────',
        'Description': '─────────────────'
    })

    # Through hidden layers
    prev_size = input_size
    for i, n_neurons in enumerate(hidden_neurons, 1):
        # Weight multiplication
        dimensions.append({
            'Stage': f'W{i} x A{i-1}',
            'Shape': f'({n_neurons}, {prev_size}) x ({prev_size}, {batch_size})',
            'Description': 'Matrix multiplication'
        })

        # Linear output
        dimensions.append({
            'Stage': f'Z{i} = W{i}A{i-1} + b{i}',
            'Shape': f'({n_neurons}, {batch_size})',
            'Description': 'Broadcast addition'
        })

        # Activation
        dimensions.append({
            'Stage': f'A{i} = ReLU(Z{i})',
            'Shape': f'({n_neurons}, {batch_size})',
            'Description': 'Element-wise'
        })

        dimensions.append({
            'Stage': '─────────',
            'Shape': '─────────',
            'Description': '─────────────────'
        })

        prev_size = n_neurons

    # Output layer
    out_idx = len(hidden_neurons) + 1
    dimensions.append({
        'Stage': f'W{out_idx} x A{len(hidden_neurons)}',
        'Shape': f'({output_size}, {prev_size}) x ({prev_size}, {batch_size})',
        'Description': 'Matrix multiplication'
    })

    dimensions.append({
        'Stage': f'Z{out_idx} = W{out_idx}A{len(hidden_neurons)} + b{out_idx}',
        'Shape': f'({output_size}, {batch_size})',
        'Description': 'Broadcast addition'
    })

    dimensions.append({
        'Stage': f'Y_pred = Softmax(Z{out_idx})',
        'Shape': f'({output_size}, {batch_size})',
        'Description': 'Column normalization'
    })

    df = pd.DataFrame(dimensions)
    print(df.to_string(index=False))
    print("=" * 80)

# Show dimensions with current configuration
show_dimensions(INPUT_SIZE, HIDDEN_NEURONS, OUTPUT_SIZE)

---

## **6.2 Matemáticas simples del forward pass**

Ahora veamos las ecuaciones matemáticas exactas de cada paso. No te preocupes, las explicaremos detalladamente.

**Notación:**
- **W**: Matriz de pesos
- **b**: Vector de bias
- **Z**: Salida lineal (antes de activación)
- **A**: Salida activada
- **m**: Número de muestras

**Ecuaciones generales para cada capa:**

```
Para capa i:
  Z_i = W_i · A_(i-1) + b_i
  A_i = ReLU(Z_i) = max(0, Z_i)

Para la capa de salida:
  Z_out = W_out · A_last + b_out
  Y_pred = Softmax(Z_out)
```

**Ejemplo con nuestra configuración actual:**

In [ ]:
# Display mathematical formulation for current configuration
def display_math_formulation(input_size, hidden_neurons, output_size):
    """
    Display the mathematical equations for the network.

    Parameters:
    - input_size: number of input features
    - hidden_neurons: list of neurons per hidden layer
    - output_size: number of output classes
    """
    print("=" * 80)
    print("MATHEMATICAL FORMULATION OF FORWARD PROPAGATION")
    print("=" * 80)

    print("\nInput:")
    print(f"  X has shape ({input_size}, m) - {input_size} features, m samples")

    # Hidden layers
    prev_size = input_size
    for i, n_neurons in enumerate(hidden_neurons, 1):
        print(f"\nHidden Layer {i}:")
        print(f"  Z{i} = W{i} · A{i-1} + b{i}")
        print(f"  A{i} = ReLU(Z{i}) = max(0, Z{i})")
        print(f"\n  Where:")
        print(f"    W{i} has shape ({n_neurons}, {prev_size})")
        print(f"    A{i-1} has shape ({prev_size}, m)" + (" - this is X" if i == 1 else ""))
        print(f"    b{i} has shape ({n_neurons}, 1)")
        print(f"    Z{i} has shape ({n_neurons}, m)")
        print(f"    A{i} has shape ({n_neurons}, m)")

        prev_size = n_neurons

    # Output layer
    out_idx = len(hidden_neurons) + 1
    print(f"\nOutput Layer:")
    print(f"  Z{out_idx} = W{out_idx} · A{len(hidden_neurons)} + b{out_idx}")
    print(f"  Y_pred = Softmax(Z{out_idx})")
    print(f"\n  Where:")
    print(f"    W{out_idx} has shape ({output_size}, {prev_size})")
    print(f"    A{len(hidden_neurons)} has shape ({prev_size}, m)")
    print(f"    b{out_idx} has shape ({output_size}, 1)")
    print(f"    Z{out_idx} has shape ({output_size}, m) - these are logits")
    print(f"    Y_pred has shape ({output_size}, m) - these are probabilities")

    print("\n" + "=" * 80)

display_math_formulation(INPUT_SIZE, HIDDEN_NEURONS, OUTPUT_SIZE)

In [ ]:
# Simple numerical example with small matrices
def forward_example():
    """
    Step-by-step numerical example of forward propagation.
    """
    print("=" * 80)
    print("NUMERICAL EXAMPLE OF FORWARD PROPAGATION")
    print("=" * 80)

    # Simplified network: 2 inputs -> 3 hidden -> 2 outputs
    # Single sample for clarity

    print("\nSimplified network: 2 inputs -> 3 hidden -> 2 outputs")
    print("Single sample for clarity\n")

    # Input
    X = np.array([[1.5],
                  [2.0]])
    print(f"Input X (2x1):\n{X}\n")

    # Layer 1: Input -> Hidden
    W1 = np.array([[0.5, -0.3],
                   [0.8,  0.2],
                   [-0.4, 0.6]])
    b1 = np.array([[0.1],
                   [-0.2],
                   [0.3]])

    print(f"W1 (3x2):\n{W1}\n")
    print(f"b1 (3x1):\n{b1}\n")

    # Forward through layer 1
    Z1 = np.dot(W1, X) + b1
    print(f"Z1 = W1 · X + b1 (3x1):\n{Z1}\n")

    A1 = relu(Z1)
    print(f"A1 = ReLU(Z1) (3x1):\n{A1}\n")

    # Layer 2: Hidden -> Output
    W2 = np.array([[0.7, -0.5, 0.3],
                   [0.4,  0.6, -0.2]])
    b2 = np.array([[0.05],
                   [-0.15]])

    print(f"W2 (2x3):\n{W2}\n")
    print(f"b2 (2x1):\n{b2}\n")

    # Forward through layer 2
    Z2 = np.dot(W2, A1) + b2
    print(f"Z2 = W2 · A1 + b2 (2x1):\n{Z2}\n")

    Y_pred = softmax(Z2)
    print(f"Y_pred = Softmax(Z2) (2x1):\n{Y_pred}\n")

    print(f"Interpretation:")
    print(f"  Probability class 0: {Y_pred[0, 0]:.4f} ({Y_pred[0, 0]*100:.2f}%)")
    print(f"  Probability class 1: {Y_pred[1, 0]:.4f} ({Y_pred[1, 0]*100:.2f}%)")
    print(f"  Prediction: Class {np.argmax(Y_pred)}")
    print(f"  Sum of probabilities: {Y_pred.sum():.6f} (should be 1.0)")
    print("=" * 80)

forward_example()

In [ ]:
# Demonstrate broadcasting
def demonstrate_broadcasting():
    """
    Show how NumPy broadcasting works when adding bias.
    """
    print("\n" + "=" * 80)
    print("BROADCASTING: How bias addition works")
    print("=" * 80)

    # Example with 3 neurons and 4 samples
    Z = np.array([[1.0, 2.0, 3.0, 4.0],
                  [0.5, 1.5, 2.5, 3.5],
                  [2.0, 3.0, 4.0, 5.0]])

    b = np.array([[0.1],
                  [0.2],
                  [0.3]])

    print(f"\nZ (3 neurons x 4 samples):\n{Z}\n")
    print(f"b (3 neurons x 1):\n{b}\n")

    result = Z + b
    print(f"Z + b (broadcasting applies b to each column):\n{result}\n")

    print("Broadcasting explanation:")
    print("  - b is automatically replicated for each sample")
    print("  - Equivalent to: Z + [b, b, b, b]")
    print("  - But much more efficient in memory and speed")
    print("=" * 80)

demonstrate_broadcasting()

---

## **6.3 Implementación con NumPy**

Ahora implementemos el forward propagation completo como funciones modulares y reutilizables.

In [ ]:
# Forward propagation implementation
def forward_propagation(X, parameters):
    """
    Perform forward propagation through the entire network.

    Parameters:
    - X: input data, shape (n_features, m_samples)
    - parameters: dictionary containing weights and biases
                  Format: {'W1': ..., 'b1': ..., 'W2': ..., 'b2': ..., ...}

    Returns:
    - Y_pred: predictions (probabilities), shape (n_classes, m_samples)
    - cache: dictionary containing intermediate values (Z and A for each layer)
              needed for backpropagation
    """
    cache = {}
    A = X
    num_layers = len(parameters) // 2  # Each layer has W and b

    # Forward through hidden layers
    for i in range(1, num_layers):
        W = parameters[f'W{i}']
        b = parameters[f'b{i}']

        Z = np.dot(W, A) + b
        A = relu(Z)

        # Store for backpropagation
        cache[f'Z{i}'] = Z
        cache[f'A{i}'] = A

    # Output layer (with Softmax)
    W_out = parameters[f'W{num_layers}']
    b_out = parameters[f'b{num_layers}']

    Z_out = np.dot(W_out, A) + b_out
    Y_pred = softmax(Z_out)

    # Store output layer
    cache[f'Z{num_layers}'] = Z_out
    cache['Y_pred'] = Y_pred

    return Y_pred, cache


def initialize_parameters(input_size, hidden_neurons, output_size, init_method='he'):
    """
    Initialize weights and biases for the network.

    Parameters:
    - input_size: number of input features
    - hidden_neurons: list of neurons per hidden layer
    - output_size: number of output classes
    - init_method: 'random', 'xavier', or 'he'

    Returns:
    - parameters: dictionary with initialized weights and biases
    """
    np.random.seed(RANDOM_SEED)
    parameters = {}

    # Build layer sizes
    layer_sizes = [input_size] + hidden_neurons + [output_size]

    # Initialize each layer
    for i in range(len(layer_sizes) - 1):
        n_input = layer_sizes[i]
        n_output = layer_sizes[i + 1]

        # Choose initialization method
        if init_method == 'he':
            # He initialization (good for ReLU)
            scale = np.sqrt(2.0 / n_input)
        elif init_method == 'xavier':
            # Xavier/Glorot initialization (good for sigmoid/tanh)
            scale = np.sqrt(1.0 / n_input)
        else:
            # Simple random initialization
            scale = 0.01

        parameters[f'W{i+1}'] = np.random.randn(n_output, n_input) * scale
        parameters[f'b{i+1}'] = np.zeros((n_output, 1))

    return parameters


# Test the implementation
print("\n" + "=" * 80)
print("TESTING FORWARD PROPAGATION")
print("=" * 80)

# Initialize parameters with current configuration
params = initialize_parameters(INPUT_SIZE, HIDDEN_NEURONS, OUTPUT_SIZE, WEIGHT_INIT)

# Test with first 3 training samples
X_sample = X_train_final[:, :3]  # Shape: (4, 3)
y_sample = y_train_one_hot[:, :3]  # Shape: (3, 3)

print(f"\nInput shape: {X_sample.shape}")
print(f"True labels shape: {y_sample.shape}\n")

# Forward pass
predictions, cache = forward_propagation(X_sample, params)

print(f"Predictions shape: {predictions.shape}")
print(f"\nPredicted probabilities (each column is a sample):\n{predictions}\n")

# Show predictions for each sample
for i in range(3):
    true_class = np.argmax(y_sample[:, i])
    pred_class = np.argmax(predictions[:, i])
    confidence = predictions[pred_class, i]

    print(f"Sample {i+1}:")
    print(f"  True class:       {true_class} ({target_names[true_class]})")
    print(f"  Predicted class:  {pred_class} ({target_names[pred_class]})")
    print(f"  Confidence:       {confidence:.4f} ({confidence*100:.2f}%)")
    print(f"  All probabilities: {predictions[:, i]}")
    print()

print("Verification:")
print(f"  - Each column sums to 1.0: {np.allclose(predictions.sum(axis=0), 1.0)}")
print(f"  - All probabilities in [0, 1]: {np.all((predictions >= 0) & (predictions <= 1))}")
print("=" * 80)

---

## **6.4 Ejemplo numérico paso a paso**

Finalmente, veamos un ejemplo completo trazando los valores a través de toda la red.

In [ ]:
# Detailed step-by-step example with a single sample
def detailed_forward_example():
    """
    Trace a single sample through the entire network with all intermediate values.
    """
    print("\n" + "=" * 80)
    print("DETAILED EXAMPLE: ONE SAMPLE THROUGH THE ENTIRE NETWORK")
    print("=" * 80)

    # Use a single sample from training set
    sample_idx = 0
    X_single = X_train_final[:, sample_idx:sample_idx+1]  # Keep 2D shape
    y_single = y_train_one_hot[:, sample_idx:sample_idx+1]

    print(f"\nSelected sample (index {sample_idx}):")
    print(f"  Normalized features: {X_single.flatten()}")
    print(f"  True label: {np.argmax(y_single)} ({target_names[np.argmax(y_single)]})\n")

    # Initialize fresh parameters for reproducibility
    params = initialize_parameters(INPUT_SIZE, HIDDEN_NEURONS, OUTPUT_SIZE, WEIGHT_INIT)

    # Forward through each layer
    A = X_single

    for layer_idx in range(1, len(HIDDEN_NEURONS) + 1):
        print("-" * 80)
        print(f"LAYER {layer_idx}: " +
              ("Input -> Hidden1" if layer_idx == 1 else f"Hidden{layer_idx-1} -> Hidden{layer_idx}"))
        print("-" * 80)

        W = params[f'W{layer_idx}']
        b = params[f'b{layer_idx}']

        Z = np.dot(W, A) + b
        A_new = relu(Z)

        print(f"W{layer_idx} shape: {W.shape}, b{layer_idx} shape: {b.shape}")
        print(f"Z{layer_idx} = W{layer_idx} · A{layer_idx-1} + b{layer_idx}:")
        print(f"{Z.flatten()}\n")
        print(f"A{layer_idx} = ReLU(Z{layer_idx}):")
        print(f"{A_new.flatten()}\n")
        print(f"Active neurons (>0): {np.sum(A_new > 0)}/{HIDDEN_NEURONS[layer_idx-1]}")

        A = A_new

    # Output layer
    out_idx = len(HIDDEN_NEURONS) + 1
    print("\n" + "-" * 80)
    print(f"LAYER {out_idx}: Hidden{len(HIDDEN_NEURONS)} -> Output")
    print("-" * 80)

    W_out = params[f'W{out_idx}']
    b_out = params[f'b{out_idx}']

    Z_out = np.dot(W_out, A) + b_out
    Y_pred = softmax(Z_out)

    print(f"W{out_idx} shape: {W_out.shape}, b{out_idx} shape: {b_out.shape}")
    print(f"Z{out_idx} = W{out_idx} · A{len(HIDDEN_NEURONS)} + b{out_idx} (logits):")
    print(f"{Z_out.flatten()}\n")
    print(f"Y_pred = Softmax(Z{out_idx}) (probabilities):")
    print(f"{Y_pred.flatten()}\n")

    print("=" * 80)
    print("FINAL RESULT")
    print("=" * 80)
    pred_class = np.argmax(Y_pred)
    true_class = np.argmax(y_single)

    for i, class_name in enumerate(target_names):
        marker = " <-- PREDICTION" if i == pred_class else ""
        true_marker = " [TRUE]" if i == true_class else ""
        print(f"  {class_name:15}: {Y_pred[i, 0]:.6f} ({Y_pred[i, 0]*100:.4f}%){marker}{true_marker}")

    print(f"\nPrediction: {target_names[pred_class]}")
    print(f"True label: {target_names[true_class]}")
    print(f"Correct: {'YES' if pred_class == true_class else 'NO'}")

    # Calculate loss
    loss = cross_entropy_loss(y_single, Y_pred)
    print(f"\nCross-Entropy Loss: {loss:.6f}")
    print("(Network is not trained yet, so prediction is random)")
    print("=" * 80)

detailed_forward_example()

In [ ]:
# Visualize activations through the network
def visualize_activations():
    """
    Visualize how activations change through network layers.
    """
    # Forward pass with full training set
    predictions, cache = forward_propagation(X_train_final, params)

    # Determine number of plots needed
    num_hidden = len(HIDDEN_NEURONS)
    fig, axes = plt.subplots(2, num_hidden + 1, figsize=(6 * (num_hidden + 1), 10))

    if num_hidden == 1:
        axes = axes.reshape(2, -1)

    # Plot hidden layer activations
    for i in range(num_hidden):
        layer_idx = i + 1
        A_layer = cache[f'A{layer_idx}']

        # Top row: activations
        im = axes[0, i].imshow(A_layer[:, :50], cmap='RdYlBu_r', aspect='auto')
        axes[0, i].set_title(f'Hidden Layer {layer_idx} Activations\n(first 50 samples)',
                            fontweight='bold', fontsize=10)
        axes[0, i].set_xlabel('Sample')
        axes[0, i].set_ylabel('Neuron')
        plt.colorbar(im, ax=axes[0, i])

        # Bottom row: activation distribution
        axes[1, i].hist(A_layer.flatten(), bins=30, alpha=0.7, edgecolor='black')
        axes[1, i].set_title(f'Hidden Layer {layer_idx}\nActivation Distribution',
                            fontweight='bold', fontsize=10)
        axes[1, i].set_xlabel('Activation Value')
        axes[1, i].set_ylabel('Frequency')
        axes[1, i].axvline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
        axes[1, i].grid(True, alpha=0.3)

    # Plot output layer
    Z_out = cache[f'Z{num_hidden + 1}']

    # Top row: output logits
    im = axes[0, num_hidden].imshow(Z_out[:, :50], cmap='RdYlBu_r', aspect='auto')
    axes[0, num_hidden].set_title('Output Logits (before Softmax)\n(first 50 samples)',
                                  fontweight='bold', fontsize=10)
    axes[0, num_hidden].set_xlabel('Sample')
    axes[0, num_hidden].set_ylabel('Class')
    axes[0, num_hidden].set_yticks(range(3))
    axes[0, num_hidden].set_yticklabels(target_names)
    plt.colorbar(im, ax=axes[0, num_hidden])

    # Bottom row: final predictions
    im = axes[1, num_hidden].imshow(predictions[:, :50], cmap='RdYlGn',
                                    aspect='auto', vmin=0, vmax=1)
    axes[1, num_hidden].set_title('Final Probabilities (after Softmax)\n(first 50 samples)',
                                  fontweight='bold', fontsize=10)
    axes[1, num_hidden].set_xlabel('Sample')
    axes[1, num_hidden].set_ylabel('Class')
    axes[1, num_hidden].set_yticks(range(3))
    axes[1, num_hidden].set_yticklabels(target_names)
    plt.colorbar(im, ax=axes[1, num_hidden])

    plt.tight_layout()
    plt.show()

    print("Observations:")
    print("  - Colors show activation intensity")
    print("  - ReLU converts negative values to 0 (dark blue)")
    print("  - Softmax normalizes probabilities (each column sums to 1)")
    print("  - Predictions are still random (network not trained yet)")

visualize_activations()

---

**Resumen de la sección:**

En esta sección has aprendido:

1. **El flujo de Forward Propagation**: Cómo los datos viajan desde la entrada hasta la salida
2. **Las matemáticas exactas**: Ecuaciones de cada capa (Z = WA + b, A = activation(Z))
3. **Implementación en NumPy**: Funciones modulares y reutilizables que se adaptan a cualquier configuración
4. **Dimensiones matriciales**: Cómo se transforman las formas en cada paso
5. **Broadcasting**: Cómo NumPy suma vectores a matrices eficientemente

Ahora que entiendes cómo la red produce predicciones, es momento de aprender cómo **mejora** esas predicciones. En la siguiente sección exploraremos el **Backward Propagation**, el algoritmo que permite a la red aprender ajustando sus pesos basándose en los errores cometidos.

# **8. Implementación Completa desde Cero**

## **8.1 Código modular con NumPy**

Ahora que entendemos todos los componentes individuales, es momento de juntar todo en una implementación completa, modular y fácil de usar. Organizaremos nuestro código de manera que sea claro, reutilizable y profesional.

**Arquitectura del código:**

```
1. Funciones de activación y sus derivadas
2. Función de pérdida
3. Inicialización de parámetros
4. Forward propagation
5. Backward propagation
6. Actualización de parámetros
7. Función de entrenamiento completa
8. Funciones de evaluación
```

**Implementación completa:**

In [ ]:
# ============================================================================
# COMPLETE NEURAL NETWORK IMPLEMENTATION FROM SCRATCH
# ============================================================================

# ----------------------------------------------------------------------------
# 1. ACTIVATION FUNCTIONS AND DERIVATIVES
# ----------------------------------------------------------------------------

def relu(Z):
    """
    ReLU activation function.

    Parameters:
    - Z: pre-activation values, any shape

    Returns:
    - activation output (element-wise max(0, Z))
    """
    return np.maximum(0, Z)


def relu_derivative(Z):
    """
    Derivative of ReLU function.

    Parameters:
    - Z: pre-activation values, any shape

    Returns:
    - derivative (1 for Z > 0, 0 otherwise)
    """
    return (Z > 0).astype(float)


def softmax(Z):
    """
    Softmax activation function.

    Parameters:
    - Z: logits, shape (n_classes, n_samples)

    Returns:
    - probability distribution, same shape as Z
    """
    # Subtract max for numerical stability
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)


# ----------------------------------------------------------------------------
# 2. LOSS FUNCTION
# ----------------------------------------------------------------------------

def cross_entropy_loss(Y_true, Y_pred):
    """
    Compute cross-entropy loss.

    Parameters:
    - Y_true: true labels (one-hot), shape (n_classes, n_samples)
    - Y_pred: predicted probabilities, same shape

    Returns:
    - average loss across all samples
    """
    m = Y_true.shape[1]

    # Clip predictions to prevent log(0)
    Y_pred_clipped = np.clip(Y_pred, 1e-10, 1 - 1e-10)

    # Compute loss
    loss = -np.sum(Y_true * np.log(Y_pred_clipped)) / m

    return loss


# ----------------------------------------------------------------------------
# 3. PARAMETER INITIALIZATION
# ----------------------------------------------------------------------------

def initialize_parameters(input_size, hidden_neurons, output_size, init_method='he'):
    """
    Initialize weights and biases for the network.

    Parameters:
    - input_size: number of input features
    - hidden_neurons: list of neurons per hidden layer
    - output_size: number of output classes
    - init_method: 'random', 'xavier', or 'he'

    Returns:
    - parameters: dictionary with initialized weights and biases
    """
    np.random.seed(RANDOM_SEED)
    parameters = {}

    # Build complete layer sizes
    layer_sizes = [input_size] + hidden_neurons + [output_size]

    # Initialize each layer
    for i in range(len(layer_sizes) - 1):
        n_input = layer_sizes[i]
        n_output = layer_sizes[i + 1]

        # Choose initialization scale
        if init_method == 'he':
            scale = np.sqrt(2.0 / n_input)
        elif init_method == 'xavier':
            scale = np.sqrt(1.0 / n_input)
        else:
            scale = 0.01

        parameters[f'W{i+1}'] = np.random.randn(n_output, n_input) * scale
        parameters[f'b{i+1}'] = np.zeros((n_output, 1))

    return parameters


# ----------------------------------------------------------------------------
# 4. FORWARD PROPAGATION
# ----------------------------------------------------------------------------

def forward_propagation(X, parameters):
    """
    Perform forward propagation through the entire network.

    Parameters:
    - X: input data, shape (n_features, n_samples)
    - parameters: dictionary with weights and biases

    Returns:
    - Y_pred: predictions (probabilities), shape (n_classes, n_samples)
    - cache: dictionary with intermediate values for backpropagation
    """
    cache = {}
    A = X
    num_layers = len(parameters) // 2

    # Forward through hidden layers
    for i in range(1, num_layers):
        W = parameters[f'W{i}']
        b = parameters[f'b{i}']

        Z = np.dot(W, A) + b
        A = relu(Z)

        cache[f'Z{i}'] = Z
        cache[f'A{i}'] = A

    # Output layer with Softmax
    W_out = parameters[f'W{num_layers}']
    b_out = parameters[f'b{num_layers}']

    Z_out = np.dot(W_out, A) + b_out
    Y_pred = softmax(Z_out)

    cache[f'Z{num_layers}'] = Z_out
    cache['Y_pred'] = Y_pred

    return Y_pred, cache


# ----------------------------------------------------------------------------
# 5. BACKWARD PROPAGATION
# ----------------------------------------------------------------------------

def backward_propagation(X, Y_true, cache, parameters):
    """
    Perform backward propagation to compute gradients.

    Parameters:
    - X: input data, shape (n_features, n_samples)
    - Y_true: true labels (one-hot), shape (n_classes, n_samples)
    - cache: dictionary with forward pass values
    - parameters: dictionary with current weights and biases

    Returns:
    - gradients: dictionary with gradients for all parameters
    """
    m = X.shape[1]
    gradients = {}
    num_layers = len(parameters) // 2

    # Output layer gradient
    Y_pred = cache['Y_pred']
    dZ = Y_pred - Y_true

    # Get previous activation
    if num_layers > 1:
        A_prev = cache[f'A{num_layers-1}']
    else:
        A_prev = X

    # Output layer parameter gradients
    gradients[f'dW{num_layers}'] = (1/m) * np.dot(dZ, A_prev.T)
    gradients[f'db{num_layers}'] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

    # Backward through hidden layers
    for layer in range(num_layers - 1, 0, -1):
        W = parameters[f'W{layer+1}']

        # Gradient flowing back
        dA_prev = np.dot(W.T, dZ)

        # Apply ReLU derivative
        Z_prev = cache[f'Z{layer}']
        dZ = dA_prev * relu_derivative(Z_prev)

        # Get activation from previous layer
        if layer > 1:
            A_prev = cache[f'A{layer-1}']
        else:
            A_prev = X

        # Compute gradients
        gradients[f'dW{layer}'] = (1/m) * np.dot(dZ, A_prev.T)
        gradients[f'db{layer}'] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

    return gradients


# ----------------------------------------------------------------------------
# 6. PARAMETER UPDATE
# ----------------------------------------------------------------------------

def update_parameters(parameters, gradients, learning_rate):
    """
    Update parameters using gradient descent.

    Parameters:
    - parameters: current weights and biases
    - gradients: computed gradients
    - learning_rate: step size for updates

    Returns:
    - updated_params: dictionary with updated parameters
    """
    updated_params = {}
    num_layers = len(parameters) // 2

    for layer in range(1, num_layers + 1):
        updated_params[f'W{layer}'] = parameters[f'W{layer}'] - learning_rate * gradients[f'dW{layer}']
        updated_params[f'b{layer}'] = parameters[f'b{layer}'] - learning_rate * gradients[f'db{layer}']

    return updated_params


# ----------------------------------------------------------------------------
# 7. EVALUATION FUNCTIONS
# ----------------------------------------------------------------------------

def predict(X, parameters):
    """
    Make predictions using trained parameters.

    Parameters:
    - X: input data, shape (n_features, n_samples)
    - parameters: trained weights and biases

    Returns:
    - predictions: predicted class labels, shape (n_samples,)
    """
    Y_pred, _ = forward_propagation(X, parameters)
    predictions = np.argmax(Y_pred, axis=0)
    return predictions


def calculate_accuracy(X, Y_true, parameters):
    """
    Calculate classification accuracy.

    Parameters:
    - X: input data, shape (n_features, n_samples)
    - Y_true: true labels (one-hot), shape (n_classes, n_samples)
    - parameters: trained weights and biases

    Returns:
    - accuracy: classification accuracy (0-1)
    """
    predictions = predict(X, parameters)
    true_labels = np.argmax(Y_true, axis=0)
    accuracy = np.mean(predictions == true_labels)
    return accuracy


print("=" * 80)
print("COMPLETE NEURAL NETWORK IMPLEMENTATION")
print("=" * 80)
print("\nAll functions defined:")
print("  - Activation functions: relu, relu_derivative, softmax")
print("  - Loss function: cross_entropy_loss")
print("  - Core functions: initialize_parameters, forward_propagation,")
print("                    backward_propagation, update_parameters")
print("  - Evaluation functions: predict, calculate_accuracy")
print("\nReady to train the network!")
print("=" * 80)

---

## **8.2 Entrenamiento de la red**

Ahora implementemos la función principal de entrenamiento que coordina todo el proceso.

In [ ]:
# ============================================================================
# TRAINING FUNCTION
# ============================================================================

def train_network(X_train, Y_train, X_test, Y_test,
                  input_size, hidden_neurons, output_size,
                  learning_rate, epochs, init_method='he',
                  print_every=50, verbose=True):
    """
    Train the neural network.

    Parameters:
    - X_train: training features, shape (n_features, n_train)
    - Y_train: training labels (one-hot), shape (n_classes, n_train)
    - X_test: test features, shape (n_features, n_test)
    - Y_test: test labels (one-hot), shape (n_classes, n_test)
    - input_size: number of input features
    - hidden_neurons: list of neurons per hidden layer
    - output_size: number of output classes
    - learning_rate: learning rate for gradient descent
    - epochs: number of training iterations
    - init_method: weight initialization method
    - print_every: print progress every N epochs
    - verbose: whether to print training progress

    Returns:
    - parameters: trained weights and biases
    - history: dictionary with training history (losses and accuracies)
    """
    # Initialize parameters
    parameters = initialize_parameters(input_size, hidden_neurons, output_size, init_method)

    # History tracking
    history = {
        'train_loss': [],
        'test_loss': [],
        'train_accuracy': [],
        'test_accuracy': []
    }

    if verbose:
        print("\n" + "=" * 80)
        print("TRAINING NEURAL NETWORK")
        print("=" * 80)
        print(f"Architecture: {input_size} -> {' -> '.join(map(str, hidden_neurons))} -> {output_size}")
        print(f"Training samples: {X_train.shape[1]}")
        print(f"Test samples: {X_test.shape[1]}")
        print(f"Learning rate: {learning_rate}")
        print(f"Epochs: {epochs}")
        print("=" * 80)
        print(f"\n{'Epoch':>6} {'Train Loss':>12} {'Test Loss':>12} {'Train Acc':>12} {'Test Acc':>12}")
        print("-" * 80)

    # Training loop
    for epoch in range(epochs):
        # Forward propagation
        Y_pred_train, cache = forward_propagation(X_train, parameters)

        # Compute loss
        train_loss = cross_entropy_loss(Y_train, Y_pred_train)

        # Backward propagation
        gradients = backward_propagation(X_train, Y_train, cache, parameters)

        # Update parameters
        parameters = update_parameters(parameters, gradients, learning_rate)

        # Calculate accuracies and test loss (every print_every epochs)
        if epoch % print_every == 0 or epoch == epochs - 1:
            train_acc = calculate_accuracy(X_train, Y_train, parameters)

            Y_pred_test, _ = forward_propagation(X_test, parameters)
            test_loss = cross_entropy_loss(Y_test, Y_pred_test)
            test_acc = calculate_accuracy(X_test, Y_test, parameters)

            # Store history
            history['train_loss'].append(train_loss)
            history['test_loss'].append(test_loss)
            history['train_accuracy'].append(train_acc)
            history['test_accuracy'].append(test_acc)

            if verbose:
                print(f"{epoch:6d} {train_loss:12.6f} {test_loss:12.6f} {train_acc:12.4f} {test_acc:12.4f}")

    if verbose:
        print("-" * 80)
        print("Training completed!")
        print("=" * 80)

    return parameters, history


# Train the network with current configuration
print("\nTraining network with current configuration...")
print(f"Hidden neurons: {HIDDEN_NEURONS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Epochs: {EPOCHS}")

trained_params, training_history = train_network(
    X_train_final, y_train_one_hot,
    X_test_final, y_test_one_hot,
    INPUT_SIZE, HIDDEN_NEURONS, OUTPUT_SIZE,
    LEARNING_RATE, EPOCHS, WEIGHT_INIT,
    print_every=50, verbose=True
)

---

## **8.3 Visualizando el aprendizaje (curva de pérdida)**

Visualizar el progreso del entrenamiento nos ayuda a entender cómo está aprendiendo la red y detectar problemas como overfitting o underfitting.

In [ ]:
# ============================================================================
# TRAINING VISUALIZATION
# ============================================================================

def plot_training_history(history, title="Training History"):
    """
    Plot training and test loss/accuracy over epochs.

    Parameters:
    - history: dictionary with training history
    - title: plot title
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Extract epoch numbers (assuming print_every was used)
    num_points = len(history['train_loss'])
    epochs_recorded = np.linspace(0, EPOCHS-1, num_points).astype(int)

    # Plot 1: Loss curves
    axes[0].plot(epochs_recorded, history['train_loss'], 'b-', linewidth=2, label='Train Loss', marker='o')
    axes[0].plot(epochs_recorded, history['test_loss'], 'r-', linewidth=2, label='Test Loss', marker='s')
    axes[0].set_xlabel('Epoch', fontsize=11)
    axes[0].set_ylabel('Loss (Cross-Entropy)', fontsize=11)
    axes[0].set_title('Loss Curves', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot 2: Accuracy curves
    axes[1].plot(epochs_recorded, history['train_accuracy'], 'b-', linewidth=2, label='Train Accuracy', marker='o')
    axes[1].plot(epochs_recorded, history['test_accuracy'], 'r-', linewidth=2, label='Test Accuracy', marker='s')
    axes[1].set_xlabel('Epoch', fontsize=11)
    axes[1].set_ylabel('Accuracy', fontsize=11)
    axes[1].set_title('Accuracy Curves', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 1.05])

    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print final metrics
    print("\n" + "=" * 80)
    print("FINAL RESULTS")
    print("=" * 80)
    print(f"Final Train Loss:     {history['train_loss'][-1]:.6f}")
    print(f"Final Test Loss:      {history['test_loss'][-1]:.6f}")
    print(f"Final Train Accuracy: {history['train_accuracy'][-1]:.4f} ({history['train_accuracy'][-1]*100:.2f}%)")
    print(f"Final Test Accuracy:  {history['test_accuracy'][-1]:.4f} ({history['test_accuracy'][-1]*100:.2f}%)")

    # Diagnose potential issues
    print("\n" + "-" * 80)
    print("DIAGNOSIS")
    print("-" * 80)

    train_acc_final = history['train_accuracy'][-1]
    test_acc_final = history['test_accuracy'][-1]
    gap = train_acc_final - test_acc_final

    if gap > 0.1:
        print("OVERFITTING detected:")
        print("  - Training accuracy much higher than test accuracy")
        print("  - Suggestions: reduce model complexity, add regularization, get more data")
    elif train_acc_final < 0.85 and test_acc_final < 0.85:
        print("UNDERFITTING detected:")
        print("  - Both training and test accuracy are low")
        print("  - Suggestions: increase model complexity, train longer, adjust learning rate")
    else:
        print("GOOD FIT:")
        print("  - Training and test accuracies are similar and high")
        print("  - Model generalizes well to unseen data")

    print("=" * 80)


# Visualize training history
plot_training_history(training_history,
                      title=f"Training History: {' -> '.join(map(str, [INPUT_SIZE] + HIDDEN_NEURONS + [OUTPUT_SIZE]))}")

**Análisis detallado de las predicciones:**

In [ ]:
# ============================================================================
# DETAILED PREDICTION ANALYSIS
# ============================================================================

def analyze_predictions(X_test, Y_test, parameters, target_names):
    """
    Detailed analysis of model predictions.

    Parameters:
    - X_test: test features
    - Y_test: test labels (one-hot)
    - parameters: trained parameters
    - target_names: list of class names
    """
    # Make predictions
    predictions = predict(X_test, parameters)
    true_labels = np.argmax(Y_test, axis=0)

    # Get prediction probabilities
    Y_pred_probs, _ = forward_propagation(X_test, parameters)

    print("\n" + "=" * 80)
    print("PREDICTION ANALYSIS")
    print("=" * 80)

    # Confusion matrix
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(true_labels, predictions)

    print("\nConfusion Matrix:")
    print("-" * 80)
    print(f"{'':15} " + " ".join([f"{name:>12}" for name in target_names]))
    print("-" * 80)
    for i, name in enumerate(target_names):
        print(f"{name:15} " + " ".join([f"{cm[i, j]:12d}" for j in range(len(target_names))]))
    print("-" * 80)

    # Classification report
    print("\nClassification Report:")
    print("-" * 80)
    from sklearn.metrics import classification_report
    print(classification_report(true_labels, predictions,
                                target_names=target_names,
                                digits=4))

    # Show some example predictions
    print("\nExample Predictions (first 10 test samples):")
    print("-" * 80)
    print(f"{'Sample':>7} {'True':>15} {'Predicted':>15} {'Confidence':>12} {'Correct':>10}")
    print("-" * 80)

    for i in range(min(10, len(predictions))):
        true_class = target_names[true_labels[i]]
        pred_class = target_names[predictions[i]]
        confidence = Y_pred_probs[predictions[i], i]
        correct = "YES" if predictions[i] == true_labels[i] else "NO"

        print(f"{i:7d} {true_class:>15} {pred_class:>15} {confidence:12.4f} {correct:>10}")

    print("-" * 80)

    # Visualize confusion matrix
    fig, ax = plt.subplots(figsize=(8, 6))

    im = ax.imshow(cm, cmap='Blues', aspect='auto')

    # Set ticks
    ax.set_xticks(range(len(target_names)))
    ax.set_yticks(range(len(target_names)))
    ax.set_xticklabels(target_names)
    ax.set_yticklabels(target_names)

    # Add text annotations
    for i in range(len(target_names)):
        for j in range(len(target_names)):
            text = ax.text(j, i, cm[i, j],
                          ha="center", va="center",
                          color="white" if cm[i, j] > cm.max() / 2 else "black",
                          fontsize=14, fontweight='bold')

    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_title('Confusion Matrix', fontsize=12, fontweight='bold')

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

    print("=" * 80)


# Analyze predictions
analyze_predictions(X_test_final, y_test_one_hot, trained_params, target_names)

**Visualización de la confianza de las predicciones:**

In [ ]:
# Visualize prediction confidence
def visualize_prediction_confidence(X_test, Y_test, parameters, target_names):
    """
    Visualize prediction confidence for test samples.

    Parameters:
    - X_test: test features
    - Y_test: test labels (one-hot)
    - parameters: trained parameters
    - target_names: list of class names
    """
    # Get predictions and probabilities
    Y_pred_probs, _ = forward_propagation(X_test, parameters)
    predictions = np.argmax(Y_pred_probs, axis=0)
    true_labels = np.argmax(Y_test, axis=0)

    # Get confidence (max probability)
    confidence = np.max(Y_pred_probs, axis=0)

    # Separate correct and incorrect predictions
    correct_mask = predictions == true_labels
    correct_confidence = confidence[correct_mask]
    incorrect_confidence = confidence[~correct_mask]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Confidence distribution
    axes[0].hist(correct_confidence, bins=15, alpha=0.7, label='Correct',
                color='green', edgecolor='black')
    if len(incorrect_confidence) > 0:
        axes[0].hist(incorrect_confidence, bins=15, alpha=0.7, label='Incorrect',
                    color='red', edgecolor='black')
    axes[0].set_xlabel('Prediction Confidence', fontsize=11)
    axes[0].set_ylabel('Frequency', fontsize=11)
    axes[0].set_title('Distribution of Prediction Confidence', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')

    # Plot 2: Confidence by class
    for class_idx, class_name in enumerate(target_names):
        class_mask = true_labels == class_idx
        class_confidence = confidence[class_mask]

        axes[1].scatter(np.full(len(class_confidence), class_idx),
                       class_confidence, alpha=0.6, s=50, label=class_name)

    axes[1].set_xticks(range(len(target_names)))
    axes[1].set_xticklabels(target_names)
    axes[1].set_ylabel('Prediction Confidence', fontsize=11)
    axes[1].set_xlabel('True Class', fontsize=11)
    axes[1].set_title('Confidence by True Class', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].set_ylim([0, 1.05])

    plt.tight_layout()
    plt.show()

    print(f"\nAverage confidence for correct predictions: {np.mean(correct_confidence):.4f}")
    if len(incorrect_confidence) > 0:
        print(f"Average confidence for incorrect predictions: {np.mean(incorrect_confidence):.4f}")
    else:
        print("No incorrect predictions!")


visualize_prediction_confidence(X_test_final, y_test_one_hot, trained_params, target_names)

---

**Resumen de la sección:**

En esta sección has aprendido:

1. **Implementación modular completa**: Todo el código organizado en funciones reutilizables
2. **Función de entrenamiento**: Coordinación de forward pass, backward pass y actualización de parámetros
3. **Visualización del aprendizaje**: Curvas de pérdida y precisión para diagnosticar el entrenamiento
4. **Análisis de predicciones**: Matriz de confusión, reporte de clasificación y confianza de predicciones

La red neuronal está ahora completamente implementada y entrenada. En la siguiente sección realizaremos experimentos comparando diferentes arquitecturas y finalmente compararemos nuestros resultados con Scikit-learn para validar nuestra implementación.

# **7. Backward Propagation: Aprendiendo del Error**

## **7.1 La intuición: "¿Quién tiene la culpa del error?"**

Imagina que eres parte de un equipo que debe lanzar una pelota a una canasta. Fallas el tiro. ¿Qué salió mal?

- ¿La persona que lanzó usó demasiada fuerza?
- ¿La persona que pasó la pelota la dirigió mal?
- ¿El ángulo inicial fue incorrecto?

**Backpropagation** hace exactamente esto: **reparte la "culpa" del error hacia atrás**, desde la salida hasta la entrada, determinando cuánto contribuyó cada parámetro (peso y bias) al error final.

**La idea central:**

```
Error en la salida → ¿Cuánto contribuyó la última capa?
                  → ¿Cuánto contribuyó la penúltima capa?
                  → ¿Cuánto contribuyó la primera capa?
```

**Analogía del restaurante:**

Pienso en una cadena de preparación de comida:

```
Ingredientes → Chef 1 → Chef 2 → Chef 3 → Plato final
                                              ↓
                                         Cliente insatisfecho
```

Si el cliente se queja, necesitas saber:
- ¿Fue culpa del Chef 3 (presentación final)?
- ¿Fue culpa del Chef 2 (cocción)?
- ¿Fue culpa del Chef 1 (preparación inicial)?
- ¿O fueron los ingredientes?

Backpropagation calcula matemáticamente cuánta "culpa" tiene cada etapa.

**Visualización conceptual:**

In [ ]:
# Visualize the backpropagation concept
def visualize_backprop_concept():
    """
    Conceptual visualization of error flowing backwards.
    """
    fig, ax = plt.subplots(figsize=(14, 8))

    # Forward pass (top half)
    forward_y = 6
    stages_forward = [
        ('Input\nX', 1), ('Hidden 1\nA1', 3), ('Hidden 2\nA2', 5),
        ('Output\nY_pred', 7), ('Loss\nL', 9)
    ]

    for name, x in stages_forward:
        circle = plt.Circle((x, forward_y), 0.4, color='lightblue',
                           ec='navy', linewidth=2, zorder=3)
        ax.add_patch(circle)
        ax.text(x, forward_y, name, ha='center', va='center',
               fontsize=9, fontweight='bold')

    # Forward arrows
    for i in range(len(stages_forward) - 1):
        x1, x2 = stages_forward[i][1], stages_forward[i+1][1]
        ax.annotate('', xy=(x2-0.5, forward_y), xytext=(x1+0.5, forward_y),
                   arrowprops=dict(arrowstyle='->', lw=2, color='green'))

    ax.text(5, forward_y + 1, 'FORWARD PASS', ha='center',
           fontsize=11, fontweight='bold', color='green')

    # Backward pass (bottom half)
    backward_y = 3
    num_hidden = len(HIDDEN_NEURONS)

    # Build backward stages dynamically
    stages_backward = [('dL/dY_pred', 9)]

    # Add gradients for each layer in reverse
    for i in range(num_hidden, 0, -1):
        stages_backward.append((f'dL/dZ{i+1}', 7 - (num_hidden - i) * 2))
        stages_backward.append((f'dL/dA{i}', 6 - (num_hidden - i) * 2))

    stages_backward.append(('dL/dZ1', 2))

    for name, x in stages_backward:
        rect = plt.Rectangle((x-0.6, backward_y-0.3), 1.2, 0.6,
                            facecolor='lightcoral', edgecolor='darkred',
                            linewidth=2, zorder=3)
        ax.add_patch(rect)
        ax.text(x, backward_y, name, ha='center', va='center',
               fontsize=8, fontweight='bold')

    # Backward arrows
    for i in range(len(stages_backward) - 1):
        x1, x2 = stages_backward[i][1], stages_backward[i+1][1]
        ax.annotate('', xy=(x2+0.6, backward_y), xytext=(x1-0.6, backward_y),
                   arrowprops=dict(arrowstyle='->', lw=2, color='red'))

    ax.text(5, backward_y - 1, 'BACKWARD PASS (Gradients)', ha='center',
           fontsize=11, fontweight='bold', color='red')

    # Show gradient flow to parameters
    param_positions = [(2, 4.5, 'W1, b1'), (4, 4.5, 'W2, b2')]
    if len(HIDDEN_NEURONS) > 1:
        param_positions.append((6, 4.5, 'W3, b3'))

    for x, y, name in param_positions:
        ax.text(x, y, name, ha='center', va='center', fontsize=9,
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
        ax.annotate('', xy=(x, y-0.3), xytext=(x, backward_y+0.4),
                   arrowprops=dict(arrowstyle='->', lw=1.5, color='orange',
                                 linestyle='dashed'))

    # Add explanation boxes
    ax.text(5, 1, 'Gradients flow backwards,\ncalculating how much each parameter\ncontributed to the error',
           ha='center', va='center', fontsize=10,
           bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.axis('off')
    ax.set_title('Backpropagation: Error Flowing Backwards',
                fontsize=13, fontweight='bold', pad=20)

    plt.tight_layout()
    plt.show()

visualize_backprop_concept()

print("The symbol dL/dX means: 'how much does loss L change when we change X'")
print("This is called a 'partial derivative' or 'gradient'")

**¿Por qué es eficiente backpropagation?**

Imagina una red con millones de parámetros. Sin backpropagation, tendrías que:
1. Cambiar cada peso individualmente
2. Recalcular el error completo
3. Repetir para TODOS los pesos

Con backpropagation:
1. Un solo pase hacia adelante
2. Un solo pase hacia atrás
3. Obtienes el gradiente de TODOS los pesos simultáneamente

---

## **7.2 La Regla de la Cadena (explicación visual y simple)**

La **Regla de la Cadena** es el concepto matemático que hace posible backpropagation. Suena complicado, pero es bastante intuitivo.

**Analogía de la cadena:**

Imagina una fábrica con tres etapas:

```
Materia prima → Proceso A → Proceso B → Proceso C → Producto final
    (x)           (f)         (g)         (h)         (y)
```

Si quieres saber cómo un pequeño cambio en la materia prima afecta al producto final, necesitas considerar TODAS las etapas intermedias.

**Matemáticamente:**

```
y = h(g(f(x)))

¿Cómo cambia y cuando cambiamos x?

dy/dx = (dy/dh) × (dh/dg) × (dg/df) × (df/dx)
```

Es como una **cadena de causas y efectos** que se multiplican.

**Ejemplo numérico simple:**

In [ ]:
# Simple chain rule example
def chain_rule_example():
    """
    Demonstrate chain rule with a simple example.
    """
    print("=" * 80)
    print("SIMPLE CHAIN RULE EXAMPLE")
    print("=" * 80)

    # Simple composite function: y = (x + 2)²
    # Let's break it down:
    #   u = x + 2    (inner function)
    #   y = u²       (outer function)

    x = 3

    # Forward pass
    u = x + 2
    y = u ** 2

    print(f"\nComposite function: y = (x + 2)²")
    print(f"\nDecomposition:")
    print(f"  u = x + 2")
    print(f"  y = u²")
    print(f"\nWith x = {x}:")
    print(f"  u = {u}")
    print(f"  y = {y}")

    # Backward pass (chain rule)
    print(f"\n{'='*80}")
    print("GRADIENT CALCULATION (dy/dx)")
    print("=" * 80)

    # dy/du = 2u
    dy_du = 2 * u
    print(f"\nStep 1: How does y change when we change u?")
    print(f"  dy/du = 2u = 2 × {u} = {dy_du}")

    # du/dx = 1
    du_dx = 1
    print(f"\nStep 2: How does u change when we change x?")
    print(f"  du/dx = 1")

    # dy/dx = dy/du × du/dx (CHAIN RULE!)
    dy_dx = dy_du * du_dx
    print(f"\nStep 3: Chain rule")
    print(f"  dy/dx = (dy/du) × (du/dx)")
    print(f"  dy/dx = {dy_du} × {du_dx} = {dy_dx}")

    print(f"\n{'='*80}")
    print("VERIFICATION")
    print("=" * 80)

    # Verify with numerical approximation
    epsilon = 0.0001
    x_plus = x + epsilon
    u_plus = x_plus + 2
    y_plus = u_plus ** 2

    numerical_gradient = (y_plus - y) / epsilon

    print(f"\nAnalytical gradient (chain rule): {dy_dx}")
    print(f"Numerical gradient (approximation): {numerical_gradient:.4f}")
    print(f"Difference:                         {abs(dy_dx - numerical_gradient):.6f}")

    print("\nInterpretation:")
    print(f"  If we increase x by 1 unit, y will increase by approximately {dy_dx} units")
    print("=" * 80)

chain_rule_example()

**Visualización gráfica de la cadena:**

In [ ]:
# Visualize chain rule graphically
def visualize_chain_rule():
    """
    Visual representation of how gradients flow through layers.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left plot: Function composition
    x = np.linspace(0, 4, 100)
    u = x + 2
    y = u ** 2

    axes[0].plot(x, u, 'b-', linewidth=2, label='u = x + 2')
    axes[0].plot(x, y, 'r-', linewidth=2, label='y = u² = (x+2)²')
    axes[0].set_xlabel('x', fontsize=11)
    axes[0].set_ylabel('Value', fontsize=11)
    axes[0].set_title('Function Composition', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Right plot: Gradient flow
    axes[1].text(0.5, 0.8, 'FORWARD PASS', ha='center', fontsize=12,
                fontweight='bold', color='green',
                transform=axes[1].transAxes)

    axes[1].text(0.5, 0.65, 'x = 3  →  u = 5  →  y = 25', ha='center',
                fontsize=10, transform=axes[1].transAxes,
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

    axes[1].text(0.5, 0.4, 'BACKWARD PASS (Gradients)', ha='center',
                fontsize=12, fontweight='bold', color='red',
                transform=axes[1].transAxes)

    axes[1].text(0.5, 0.25, 'dy/dx = 10  ←  dy/du = 10, du/dx = 1',
                ha='center', fontsize=10, transform=axes[1].transAxes,
                bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))

    axes[1].text(0.5, 0.1, 'Chain Rule:\ndy/dx = (dy/du) × (du/dx) = 10 × 1 = 10',
                ha='center', fontsize=9, transform=axes[1].transAxes,
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

    axes[1].axis('off')
    axes[1].set_title('Gradient Flow', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

visualize_chain_rule()

**En redes neuronales:**

En nuestra red, la regla de la cadena se aplica así:

```
Para actualizar W1 (primera capa), necesitamos:

dL/dW1 = dL/dY_pred × dY_pred/dZ_out × dZ_out/dA_last × ... × dZ1/dW1

¡Es una cadena larga! Pero cada derivada individual es simple.
```

---

## **7.3 Calculando gradientes capa por capa**

Ahora calculemos los gradientes concretos para nuestra red.

**Notación:**
- **L**: Loss (Cross-Entropy)
- **dL/dX**: Gradiente de la pérdida respecto a X

**Capa de Salida (más fácil, comenzamos aquí):**

```
Loss = -Σ yi log(ŷi)    (Cross-Entropy)
Y_pred = Softmax(Z_out)

Gradiente (combinando Softmax + Cross-Entropy):
dL/dZ_out = Y_pred - Y_true

¡Esta es una simplificación hermosa! La derivada combinada es simplemente:
(predicción - verdad)
```

**Capas Ocultas (general):**

```
Para cualquier capa i:

Z_i = W_i · A_(i-1) + b_i
A_i = ReLU(Z_i)

Gradientes:
dL/dW_i = (dL/dZ_i) · A_(i-1)ᵀ
dL/db_i = Σ(dL/dZ_i)  [suma por columnas]
dL/dA_(i-1) = W_iᵀ · (dL/dZ_i)

Para propagar hacia atrás:
dL/dZ_(i-1) = (dL/dA_(i-1)) ⊙ ReLU'(Z_(i-1))
donde ⊙ es multiplicación element-wise
y ReLU'(z) = 1 si z > 0, else 0
```

**Implementación paso a paso:**

In [ ]:
# Backward propagation implementation
def backward_propagation(X, Y_true, cache, parameters):
    """
    Perform backward propagation to compute gradients.

    Parameters:
    - X: input data, shape (n_features, m_samples)
    - Y_true: true labels (one-hot), shape (n_classes, m_samples)
    - cache: dictionary with forward pass values (from forward_propagation)
    - parameters: dictionary with current weights and biases

    Returns:
    - gradients: dictionary containing gradients for all parameters
                 Format: {'dW1': ..., 'db1': ..., 'dW2': ..., 'db2': ..., ...}
    """
    m = X.shape[1]  # Number of samples
    gradients = {}

    num_layers = len(parameters) // 2

    # ========================================================================
    # OUTPUT LAYER
    # ========================================================================
    # Gradient of loss w.r.t Z_out (combined Softmax + Cross-Entropy derivative)
    Y_pred = cache['Y_pred']
    dZ = Y_pred - Y_true  # Shape: (n_classes, m)

    # Get previous activation (last hidden layer)
    if num_layers > 1:
        A_prev = cache[f'A{num_layers-1}']
    else:
        A_prev = X

    # Gradients for output layer
    gradients[f'dW{num_layers}'] = (1/m) * np.dot(dZ, A_prev.T)
    gradients[f'db{num_layers}'] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

    # ========================================================================
    # HIDDEN LAYERS (backward through each layer)
    # ========================================================================
    for layer in range(num_layers - 1, 0, -1):
        # Get current layer's weight matrix
        W = parameters[f'W{layer+1}']

        # Gradient flowing back to previous activation
        dA_prev = np.dot(W.T, dZ)

        # Apply ReLU derivative
        Z_prev = cache[f'Z{layer}']
        dZ = dA_prev * relu_derivative(Z_prev)

        # Get activation from layer before this one
        if layer > 1:
            A_prev = cache[f'A{layer-1}']
        else:
            A_prev = X

        # Compute gradients for this layer
        gradients[f'dW{layer}'] = (1/m) * np.dot(dZ, A_prev.T)
        gradients[f'db{layer}'] = (1/m) * np.sum(dZ, axis=1, keepdims=True)

    return gradients


# Test backward propagation
print("=" * 80)
print("TESTING BACKWARD PROPAGATION")
print("=" * 80)

# Use the same sample data from forward pass
X_sample = X_train_final[:, :3]
y_sample = y_train_one_hot[:, :3]

# Forward pass
predictions, cache = forward_propagation(X_sample, params)

# Backward pass
grads = backward_propagation(X_sample, y_sample, cache, params)

print("\nShapes of computed gradients:")
print("-" * 80)
for key in sorted(grads.keys()):
    param_key = key[1:]  # Remove 'd' prefix
    param_shape = params[param_key].shape
    grad_shape = grads[key].shape
    print(f"{key:6} shape: {str(grad_shape):12} (should match {param_key} shape: {param_shape})")

print("\n" + "=" * 80)
print("EXAMPLE VALUES (Gradients for W1)")
print("=" * 80)
print(f"\ndW1 (first 2 rows, all columns):")
print(grads['dW1'][:2, :])
print(f"\nInterpretation:")
print(f"  - Positive values: increasing this weight INCREASES loss")
print(f"  - Negative values: increasing this weight DECREASES loss")
print(f"  - Values near 0: this weight has little effect on loss")
print("=" * 80)

**Ejemplo numérico detallado:**

In [ ]:
# Detailed backward pass example
def detailed_backward_example():
    """
    Step-by-step backward propagation for a single sample.
    """
    print("\n" + "=" * 80)
    print("DETAILED BACKWARD PROPAGATION EXAMPLE")
    print("=" * 80)

    # Use single sample
    X_single = X_train_final[:, 0:1]
    y_single = y_train_one_hot[:, 0:1]

    # Forward pass
    Y_pred, cache = forward_propagation(X_single, params)

    # Calculate loss
    loss = cross_entropy_loss(y_single, Y_pred)

    print(f"\nInitial loss: {loss:.6f}")
    print(f"Prediction: {Y_pred.flatten()}")
    print(f"True:       {y_single.flatten()}\n")

    m = 1  # One sample
    num_layers = len(params) // 2

    print("-" * 80)
    print(f"OUTPUT LAYER (Layer {num_layers}) - Gradients")
    print("-" * 80)

    dZ = Y_pred - y_single
    print(f"dZ{num_layers} = Y_pred - Y_true = {dZ.flatten()}")
    print(f"  -> This is the error in prediction")
    print(f"  -> Positive values: over-prediction")
    print(f"  -> Negative values: under-prediction\n")

    # Get previous activation
    if num_layers > 1:
        A_prev = cache[f'A{num_layers-1}']
    else:
        A_prev = X_single

    dW = np.dot(dZ, A_prev.T)
    db = np.sum(dZ, axis=1, keepdims=True)
    print(f"dW{num_layers} shape: {dW.shape}")
    print(f"db{num_layers} = {db.flatten()}\n")

    # Backward through hidden layers
    for layer in range(num_layers - 1, 0, -1):
        print("-" * 80)
        print(f"HIDDEN LAYER {layer} - Gradients")
        print("-" * 80)

        W = params[f'W{layer+1}']
        dA = np.dot(W.T, dZ)
        print(f"dA{layer} (gradient flowing back): {dA.flatten()}")

        Z = cache[f'Z{layer}']
        relu_deriv = relu_derivative(Z)
        print(f"ReLU'(Z{layer}) (binary mask): {relu_deriv.flatten()}")
        print(f"  -> 1 where neuron was active, 0 where inactive")

        dZ = dA * relu_deriv
        print(f"dZ{layer} (after applying ReLU'): {dZ.flatten()}\n")

    print("=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print("\nGradient flows backwards:")
    print("  1. Starts with error at output (Y_pred - Y_true)")
    print("  2. Propagates through each layer by multiplying by:")
    print("     a) Transposed weights (W^T)")
    print("     b) Derivative of activation function")
    print("  3. At each layer, computes gradients for W and b")
    print("\nThese gradients indicate the direction to REDUCE loss")
    print("=" * 80)

detailed_backward_example()

---

## **7.4 Actualizando pesos: El descenso del gradiente**

Una vez que tenemos los gradientes, podemos **actualizar** los parámetros para reducir la pérdida.

**Regla de actualización:**

```
W_nuevo = W_actual - learning_rate × dL/dW
b_nuevo = b_actual - learning_rate × dL/db
```

El **signo negativo** es crucial: nos movemos en la dirección **opuesta** al gradiente para minimizar la pérdida.

**Implementación:**

In [ ]:
# Update parameters using gradients
def update_parameters(parameters, gradients, learning_rate):
    """
    Update parameters using gradient descent.

    Parameters:
    - parameters: current weights and biases
    - gradients: computed gradients from backward propagation
    - learning_rate: step size for updates

    Returns:
    - updated_params: dictionary with updated parameters
    """
    updated_params = {}
    num_layers = len(parameters) // 2

    # Update each parameter
    for layer in range(1, num_layers + 1):
        updated_params[f'W{layer}'] = parameters[f'W{layer}'] - learning_rate * gradients[f'dW{layer}']
        updated_params[f'b{layer}'] = parameters[f'b{layer}'] - learning_rate * gradients[f'db{layer}']

    return updated_params


# Demonstrate parameter update
print("\n" + "=" * 80)
print("PARAMETER UPDATE")
print("=" * 80)

# Show W1 before update
print(f"\nW1 before update (first 2 rows):")
print(params['W1'][:2, :])

# Update parameters
params_updated = update_parameters(params, grads, LEARNING_RATE)

# Show W1 after update
print(f"\nW1 after update (first 2 rows):")
print(params_updated['W1'][:2, :])

# Show the change
print(f"\nChange in W1 (first 2 rows):")
change = params_updated['W1'][:2, :] - params['W1'][:2, :]
print(change)

print(f"\nLearning rate used: {LEARNING_RATE}")
print(f"Change = -learning_rate × gradient")
print("=" * 80)

**Visualización del descenso del gradiente:**

In [ ]:
# Visualize gradient descent on a simple loss surface
def visualize_gradient_descent():
    """
    Show how gradient descent navigates the loss landscape.
    """
    # Create a simple 2D loss surface (parabola)
    w1_range = np.linspace(-3, 3, 100)
    w2_range = np.linspace(-3, 3, 100)
    W1, W2 = np.meshgrid(w1_range, w2_range)

    # Loss function: L = w1² + w2²
    Loss = W1**2 + W2**2

    # Gradient descent trajectory
    learning_rate = 0.3
    w1, w2 = 2.5, 2.5  # Starting point
    trajectory = [(w1, w2)]

    for _ in range(15):
        # Gradients: dL/dw1 = 2*w1, dL/dw2 = 2*w2
        grad_w1 = 2 * w1
        grad_w2 = 2 * w2

        # Update
        w1 = w1 - learning_rate * grad_w1
        w2 = w2 - learning_rate * grad_w2

        trajectory.append((w1, w2))

    trajectory = np.array(trajectory)

    # Plot
    fig = plt.figure(figsize=(14, 6))

    # 3D surface plot
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.plot_surface(W1, W2, Loss, cmap='viridis', alpha=0.6)

    # Plot trajectory on surface
    traj_loss = [w1**2 + w2**2 for w1, w2 in trajectory]
    ax1.plot(trajectory[:, 0], trajectory[:, 1], traj_loss,
            'ro-', linewidth=2, markersize=6, label='Gradient Descent Path')
    ax1.scatter([trajectory[0, 0]], [trajectory[0, 1]], [traj_loss[0]],
               color='green', s=100, label='Start')
    ax1.scatter([trajectory[-1, 0]], [trajectory[-1, 1]], [traj_loss[-1]],
               color='red', s=100, label='End')

    ax1.set_xlabel('w1')
    ax1.set_ylabel('w2')
    ax1.set_zlabel('Loss')
    ax1.set_title('Loss Surface with Gradient Descent Trajectory', fontweight='bold')
    ax1.legend()

    # Contour plot
    ax2 = fig.add_subplot(122)
    contour = ax2.contour(W1, W2, Loss, levels=20, cmap='viridis')
    ax2.clabel(contour, inline=True, fontsize=8)

    # Plot trajectory
    ax2.plot(trajectory[:, 0], trajectory[:, 1], 'ro-', linewidth=2,
            markersize=6, label='Optimization Path')
    ax2.scatter([trajectory[0, 0]], [trajectory[0, 1]],
               color='green', s=100, zorder=5, label='Start')
    ax2.scatter([trajectory[-1, 0]], [trajectory[-1, 1]],
               color='red', s=100, zorder=5, label='End')
    ax2.scatter([0], [0], color='gold', s=150, marker='*',
               zorder=5, label='Optimal (0,0)')

    ax2.set_xlabel('w1')
    ax2.set_ylabel('w2')
    ax2.set_title('Top View (Contour Plot)', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("Observe how gradient descent:")
    print("  - Starts far from minimum (green)")
    print("  - Takes steps in the direction of steepest descent")
    print("  - Converges toward global minimum (gold star)")
    print(f"  - Final position: ({trajectory[-1, 0]:.4f}, {trajectory[-1, 1]:.4f})")

visualize_gradient_descent()

**El ciclo completo de entrenamiento:**

In [ ]:
# Complete training iteration
def training_iteration_demo():
    """
    Demonstrate one complete iteration of training.
    """
    print("\n" + "=" * 80)
    print("ONE COMPLETE TRAINING CYCLE")
    print("=" * 80)

    # Initialize fresh parameters
    params_demo = initialize_parameters(INPUT_SIZE, HIDDEN_NEURONS, OUTPUT_SIZE, WEIGHT_INIT)

    # Use small batch
    X_batch = X_train_final[:, :10]
    y_batch = y_train_one_hot[:, :10]

    print("\n[1] Forward Propagation")
    print("-" * 80)
    Y_pred, cache = forward_propagation(X_batch, params_demo)
    loss_before = cross_entropy_loss(y_batch, Y_pred)
    print(f"Loss before update: {loss_before:.6f}")

    print("\n[2] Backward Propagation")
    print("-" * 80)
    gradients = backward_propagation(X_batch, y_batch, cache, params_demo)
    print("Gradients computed for all parameters")

    print("\n[3] Parameter Update")
    print("-" * 80)
    params_updated = update_parameters(params_demo, gradients, LEARNING_RATE)
    print(f"Parameters updated with learning rate = {LEARNING_RATE}")

    print("\n[4] Verification")
    print("-" * 80)
    Y_pred_after, _ = forward_propagation(X_batch, params_updated)
    loss_after = cross_entropy_loss(y_batch, Y_pred_after)
    print(f"Loss after update: {loss_after:.6f}")
    print(f"Loss reduction: {loss_before - loss_after:.6f}")

    if loss_after < loss_before:
        print("\nSuccess! Loss decreased -> Network is learning")
    else:
        print("\nWarning: Loss increased -> Possible learning rate too high")

    print("=" * 80)

training_iteration_demo()

---

**Resumen de la sección:**

En esta sección has aprendido:

1. **La intuición de backpropagation**: Repartir la "culpa" del error hacia atrás
2. **La Regla de la Cadena**: El principio matemático que hace posible calcular gradientes eficientemente
3. **Cálculo de gradientes**: Derivadas específicas para cada capa (Softmax + Cross-Entropy, ReLU, transformaciones lineales)
4. **Descenso del gradiente**: Cómo usar los gradientes para actualizar parámetros y reducir la pérdida

Ahora comprendes tanto el **forward pass** (cómo la red hace predicciones) como el **backward pass** (cómo la red aprende de sus errores). En la siguiente sección, juntaremos todo en una implementación completa y entrenaremos nuestra red neuronal desde cero para clasificar las flores del dataset Iris.